# Algorithm23 Tutorial Audit — CrystalFormer-MCMC q Refinement for KLDM-CPS-Wyckoff Sampling

This notebook is a lightweight, laptop-friendly feasibility notebook for Algorithm23. It keeps the stable KLDM-CPS/PyXtal machinery from Algorithm22 and adds a local CrystalFormer trust-region MCMC around the KLDM geometry-fitted Wyckoff `q`.

The main questions are:
- does CrystalFormer provide useful local q information inside the KLDM geometry basin?
- does that local q improvement survive the weak CPS state update?
- does Algorithm23 beat the current Algorithm22 geometry-only baseline on best-step or repeated-seed metrics?


## CrystalFormer weights

Recommended reference links:
- [CrystalFormer available weights](https://github.com/deepmodeling/CrystalFormer#available-weights)
- [CrystalFormer CSP Colab](https://colab.research.google.com/drive/1I7b5exbB2oBjexFIEaeDQexmYRDgLHVk?authuser=0#scrollTo=kfu6Ez9e6Sp7)

The CSP-only checkpoint expected by this notebook is:
- `epoch_046000.pkl`

You can either:
- set `KLDM_ALGO21_CF_CHECKPOINT` to a local checkpoint path
- or run the download cell below


## CrystalFormer dependency bootstrap

**Purpose:** CrystalFormer needs a small JAX-side stack before the checkpoint wrapper can load. If the preflight below reports missing packages such as `haiku`, run the optional install cell once, restart the kernel, and rerun from the top.


In [1]:
# Optional dependency bootstrap for CrystalFormer.
# Run this only if the dependency preflight reports missing packages.

# %pip install -U "jax[cpu]"
# %pip install dm-haiku==0.0.15 optax==0.2.6 pyxtal==1.1.1 gdown


In [2]:
import importlib.util
import pandas as pd
rows = []
for module_name in ('jax', 'haiku', 'optax', 'pyxtal', 'gdown'):
    rows.append({
        'test': 'algorithm22_cf_dependency_preflight',
        'module': module_name,
        'available': bool(importlib.util.find_spec(module_name) is not None),
    })

dep_df = pd.DataFrame(rows)
display(dep_df)


,test,module,available
0,algorithm22_cf_dependency_preflight,jax,True
1,algorithm22_cf_dependency_preflight,haiku,True
2,algorithm22_cf_dependency_preflight,optax,True
3,algorithm22_cf_dependency_preflight,pyxtal,True
4,algorithm22_cf_dependency_preflight,gdown,True


In [3]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ["JAX_DISABLE_JIT"] = str(os.environ.get("KLDM_ALGO23_DISABLE_JIT", "false"))
os.environ["KLDM_ALGO21_SAFE_MODE"] = str(os.environ.get("KLDM_ALGO23_CF_SAFE_MODE", "false"))
os.environ.setdefault("KLDM_ALGO23_ENABLE_CF_KERNEL_FALLBACK", "false")
os.environ.setdefault("KLDM_ALGO23_CF_RUNTIME_MODE", "subprocess_primary")
import time
import math
import traceback
import subprocess
import sys
import pickle
import json as jsonlib
import importlib
import gc
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from pymatgen.core import Element

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.symmetry.crystalformer_backend as cf_backend
import kldmPlus.algorithm22_faithful_kldm_cps_csp as algo22_backend
import kldmPlus.algorithm23_crystalformer_mcmc_q_refinement as algo23_backend
cf_backend = importlib.reload(cf_backend)
algo22_backend = importlib.reload(algo22_backend)
algo23_backend = importlib.reload(algo23_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import build_symmetry_frame_bridge, estimate_semantic_transport_for_reference_order, oracle_spacegroup_from_case
from kldmPlus.utils.time import iter_sampling_times, make_times
from kldmPlus.symmetry.wyckoff_templates import extract_wyckoff_templates, flatten_site_signature, sample_random_free_vars, recover_template_free_vars_from_anchor_entries

Algorithm19State = algo22_backend.Algorithm19State
build_oracle_diffcsppp_payload_from_structure = algo22_backend.build_oracle_diffcsppp_payload_from_structure
map_model_to_payload_reference_chart = algo22_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo22_backend.map_payload_reference_chart_to_model_frame
kldm_clean_fractional_denoiser_Df = algo22_backend.kldm_clean_fractional_denoiser_Df
wrap01 = algo22_backend.wrap01
wrapdiff = algo22_backend.wrapdiff
torus_rmse = algo22_backend.torus_rmse

CrystalFormerLikelihood = algo22_backend.CrystalFormerLikelihood
predict_clean_f0 = algo22_backend.predict_clean_f0
model_to_payload = algo22_backend.model_to_payload
payload_to_model = algo22_backend.payload_to_model
expand_q = algo22_backend.expand_q
fit_q_to_clean_prediction = algo22_backend.fit_q_to_clean_prediction
torus_soft_project = algo22_backend.torus_soft_project
renoise_from_f0 = algo22_backend.renoise_from_f0
sample_q_from_crystalformer = algo22_backend.sample_q_from_crystalformer
build_payload_from_template_q = algo22_backend.build_payload_from_template_q
species_match_reorder = algo22_backend.species_match_reorder
Algorithm21Config = cf_backend.Algorithm21Config
q_only_clean_cf_fit = cf_backend.q_only_clean_cf_fit
q_only_clean_cf_local_rerank = cf_backend.q_only_clean_cf_local_rerank
torus_interp = cf_backend.torus_interp
rank_q_candidates = getattr(cf_backend, 'rank_q_candidates', None)
build_crystalformer_reduced_sequence = getattr(cf_backend, 'build_crystalformer_reduced_sequence', None)
expand_crystalformer_reduced_sequence = getattr(cf_backend, 'expand_crystalformer_reduced_sequence', None)
crystalformer_reduced_sequence_debug = getattr(cf_backend, 'crystalformer_reduced_sequence_debug', None)
crystalformer_site_representative_search = getattr(cf_backend, 'crystalformer_site_representative_search', None)
crystalformer_payload_order_assembly_debug = getattr(cf_backend, 'crystalformer_payload_order_assembly_debug', None)

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO21_PROFILE', 'laptop').strip().lower()
def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def notebook_log(message: str) -> None:
    text = str(message)
    print(text)
    with ALGO21_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(text)
        if not text.endswith("\n"):
            handle.write("\n")

def cf_nll_eval(*, payload, q, lattice_feature, formula, label: str) -> float:
    notebook_log(f"[cf-eval] start {label}")
    if not CF_READY:
        notebook_log(f"[cf-eval] skip {label} CF unavailable")
        return float('nan')
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    try:
        q_np = np.asarray(torch.as_tensor(q).detach().cpu(), dtype=float)
        l_np = np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float)
        if use_subprocess:
            tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
            tmp_dir.mkdir(parents=True, exist_ok=True)
            safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
            in_path = tmp_dir / f'{safe_label}.pkl'
            out_path = tmp_dir / f'{safe_label}.json'
            with in_path.open('wb') as handle:
                pickle.dump({
                    'repo_root': str(ROOT),
                    'checkpoint_path': str(CF_CHECKPOINT),
                    'symmetry_payload': payload,
                    'q': q_np,
                    'lattice_feature': l_np,
                    'formula': formula,
                }, handle)
            cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_score_once.py'), '--input', str(in_path), '--output', str(out_path)]
            notebook_log(f"[cf-eval] subprocess start {label}")
            completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
            result = {}
            if out_path.exists():
                with out_path.open('r', encoding='utf-8') as handle:
                    result = jsonlib.load(handle)
            if completed.returncode != 0 or not result.get('ok'):
                notebook_log(f"[cf-eval] subprocess ERROR {label} returncode={completed.returncode} stderr_tail={completed.stderr[-500:]}")
                raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer subprocess failed')
            value = float(result['value'])
        else:
            raise RuntimeError('In-kernel CrystalFormer evaluation is disabled for Algorithm22 notebook stability.')
        notebook_log(f"[cf-eval] done {label} value={value:.6g}")
        return value
    except Exception as exc:
        notebook_log(f"[cf-eval] ERROR {label} {type(exc).__name__}: {exc}")
        raise
    finally:
        notebook_cleanup()

def cf_sequence_smoke(*, payload, q, lattice_feature, label: str) -> dict[str, Any]:
    notebook_log(f"[cf-seq] start {label}")
    seq = build_crystalformer_reduced_sequence(payload=payload, q=q, lattice_feature=lattice_feature)
    notebook_log(f"[cf-seq] done {label} num_sites={int(seq['num_sites'])}")
    notebook_cleanup()
    return seq

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO21_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO21_GRAPH_IDS', '2', '2'))
CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_CORE_T', '0.1,0.3,0.5', '0.1,0.3,0.5')))
BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_BETA_VALUES', '0,0.001,0.01,0.1,1,10', '0,0.001,0.01,0.1,1,10')))
ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_ALPHA_VALUES', '0,0.1,0.25,0.5,1.0', '0,0.1,0.25,0.5,1.0')))
Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_Q_OPT_STEPS', '50', '50'))
TEST45_BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_BETA_VALUES', '0', '0')))
TEST6_ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST6_ALPHA_VALUES', '0,0.25', '0,0.25')))
TEST46_Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_TEST46_Q_OPT_STEPS', '4', '4'))
TEST46_CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST46_CORE_T', '0.5', '0.5')))
TEST7_RERANK_RADIUS = float(profile_default('KLDM_ALGO21_TEST7_RERANK_RADIUS', '0.05', '0.05'))
TEST7_RERANK_CANDIDATES = int(profile_default('KLDM_ALGO21_TEST7_RERANK_CANDIDATES', '3', '3'))
TEST7_RERANK_KEEP_TOL = float(profile_default('KLDM_ALGO21_TEST7_RERANK_KEEP_TOL', '0.05', '0.05'))
TEST45_CF_SCALE_BETAS = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_CF_SCALE_BETAS', '0,1e-6,1e-5,1e-4', '0,1e-6,1e-5,1e-4')))
# CF-gradient optimization is deliberately opt-in; default notebook tests use value-only reranking.
os.environ.setdefault('KLDM_ALGO21_ENABLE_CF_GRAD', 'false')
SHORT_SAMPLER_STEPS = int(profile_default('KLDM_ALGO21_SHORT_SAMPLER_STEPS', '100', '100'))
SHORT_SAMPLER_TSTART = float(profile_default('KLDM_ALGO21_SHORT_SAMPLER_TSTART', '0.5', '0.5'))
NOTEBOOK_SAFE_MODE = str(os.environ.get("KLDM_ALGO21_SAFE_MODE", "true")).strip().lower() in {"1", "true", "yes", "on"}
SHOW_FULL_DEBUG_TABLES = not NOTEBOOK_SAFE_MODE
CF_RANDOM_TRIALS = int(profile_default('KLDM_ALGO21_CF_RANDOM_TRIALS', '1' if NOTEBOOK_SAFE_MODE else '8', '1'))
CF_CHECKPOINT = str(profile_default('KLDM_ALGO21_CF_CHECKPOINT', str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl')), str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl'))))
CF_FORMULA = os.environ.get('KLDM_ALGO21_FORMULA', '').strip() or None
ALGO21_LOG_PATH = ROOT / 'notebooks' / 'log.txt'
os.environ["KLDM_ALGO21_LOG_PATH"] = str(ALGO21_LOG_PATH)
ALGO21_LOG_PATH.write_text("", encoding="utf-8")

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

print(f'profile={PROFILE} graphs={GRAPH_IDS} t={CORE_T_VALUES} beta={BETA_VALUES} alpha={ALPHA_VALUES} q_opt_steps={Q_OPT_STEPS} cf_ckpt={CF_CHECKPOINT}')


def notebook_cleanup():
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    if use_subprocess:
        try:
            cf_obj = globals().get('cf_like', None)
            if cf_obj is not None and hasattr(cf_obj, 'release_runtime'):
                cf_obj.release_runtime()
        except Exception:
            pass
    try:
        worker = globals().get('CF_WORKER_PROC', None)
        if worker is not None:
            try:
                if worker.stdin is not None:
                    worker.stdin.write(jsonlib.dumps({'cmd': 'shutdown'}) + '\n')
                    worker.stdin.flush()
            except Exception:
                pass
            try:
                worker.wait(timeout=5)
            except Exception:
                try:
                    worker.terminate()
                except Exception:
                    pass
        globals()['CF_WORKER_PROC'] = None
    except Exception:
        pass
    gc.collect()
    try:
        import jax
        jax.clear_caches()
    except Exception:
        pass
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

Algorithm22Config = algo22_backend.Algorithm22Config
algorithm22_projection_schedule = algo22_backend.algorithm22_projection_schedule
algorithm22_piecewise_alpha = algo22_backend.algorithm22_piecewise_alpha
algorithm22_cps_gamma = algo22_backend.algorithm22_cps_gamma
algorithm22_cps_alpha = algo22_backend.algorithm22_cps_alpha
clean_fractional_estimate_22 = algo22_backend.clean_fractional_estimate
clean_lattice_estimate_22 = algo22_backend.clean_lattice_estimate
decode_state_cell_matrix_22 = algo22_backend.decode_state_cell_matrix
oracle_template_projection_22 = algo22_backend.oracle_template_projection
generate_candidates_22 = algo22_backend.generate_candidates
generate_pyxtal_candidates_22 = algo22_backend.generate_pyxtal_candidates
rank_candidates_against_clean_estimate_22 = algo22_backend.rank_candidates_against_clean_estimate
algorithm22b_ranked_kldm_cps_step = algo22_backend.algorithm22b_ranked_kldm_cps_step
group_witness_from_candidates_22 = algo22_backend.group_witness_from_candidates


Algorithm22RunResult = algo22_backend.Algorithm22RunResult
algorithm22A_oracle_template_kldm_cps = algo22_backend.algorithm22A_oracle_template_kldm_cps
algorithm22B_ranked_kldm_cps = algo22_backend.algorithm22B_ranked_kldm_cps
kldm_clean_fractional_denoiser_22 = algo22_backend.kldm_clean_fractional_denoiser
kldm_clean_lattice_denoiser_22 = algo22_backend.kldm_clean_lattice_denoiser
expand_template_to_model_order_22 = algo22_backend.expand_template_to_model_order
materialize_candidate_from_template_22 = algo22_backend.materialize_candidate_from_template
rank_projected_candidates_with_rule_22 = algo22_backend.rank_projected_candidates_with_rule
sample_template_q_proposals_22 = algo22_backend.sample_template_q_proposals
safety_ok_22 = algo22_backend.safety_ok
torus_distance_squared_22 = algo22_backend.torus_distance_squared
oracle_template_witness_22 = algo22_backend.oracle_template_witness
project_candidates_to_clean_estimate_22 = algo22_backend.project_candidates_to_clean_estimate
algorithm22_piecewise_alpha = algo22_backend.algorithm22_piecewise_alpha
algorithm22_piecewise_beta = algo22_backend.algorithm22_piecewise_beta
kldm_cps_update_or_renoise_22 = algo22_backend.kldm_cps_update_or_renoise
tdm_velocity_sigma_at_state_22 = algo22_backend.tdm_velocity_sigma_at_state
tdm_residual_epsilon_r_22 = algo22_backend.tdm_residual_epsilon_r

Algorithm23Config = algo23_backend.Algorithm23Config
Algorithm23MCMCConfig = algo23_backend.Algorithm23MCMCConfig
Algorithm23TrustRegionConfig = algo23_backend.Algorithm23TrustRegionConfig
algorithm23_refine_projected_candidate = algo23_backend.algorithm23_refine_projected_candidate
algorithm23_state_update = algo23_backend.algorithm23_state_update
algorithm23_soft_project_clean_estimate = algo23_backend.algorithm23_soft_project_clean_estimate
algorithm23_geometry_cap = algo23_backend.algorithm23_geometry_cap
algorithm23_geometry_cost = algo23_backend.algorithm23_geometry_cost


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
profile=laptop graphs=[2] t=(0.1, 0.3, 0.5) beta=(0.0, 0.001, 0.01, 0.1, 1.0, 10.0) alpha=(0.0, 0.1, 0.25, 0.5, 1.0) q_opt_steps=50 cf_ckpt=/workspace/notebooks/epoch_

In [4]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int

result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    cols = list(df.columns)
    for key in by:
        if key not in cols:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    from collections import Counter
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out

GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    l = case.gt_l_feature.detach().clone().reshape(1, -1).to(device=device, dtype=case.gt_l_feature.dtype if ref_tensor is None else case.gt_l_feature.dtype)
    atomic_numbers = case.atomic_numbers.detach().clone().to(device=device, dtype=torch.long)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    space_group = torch.tensor([int(case.requested_sg)], device=device, dtype=torch.long)

    class _SingleGraphBatch(SimpleNamespace):
        def to(self, device):
            out = _SingleGraphBatch()
            for key, value in self.__dict__.items():
                if torch.is_tensor(value):
                    setattr(out, key, value.to(device))
                else:
                    setattr(out, key, value)
            return out

    return _SingleGraphBatch(
        pos=pos,
        l=l,
        atomic_numbers=atomic_numbers,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
        space_group=space_group,
    )


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(vanilla_structure=structure, standardization='refined', symprec=1e-2, angle_tolerance=5.0)
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'transport_rmse': float(rmse_ref),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    standardized_frac_rmse = None
    if getattr(result, 'matcher_diagnostics', None) is not None:
        standardized_frac_rmse = getattr(result.matcher_diagnostics, 'standardized_frac_rmse', None)
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'composition_match': result.composition_match,
        'requested_space_group_match': result.requested_space_group_match,
        'validity_reason': result.validity_reason,
        'standardized_frac_rmse': standardized_frac_rmse,
        'metric_source': 'sample_evaluation.evaluate_csp_reconstruction / StructureMatcher',
    }


def algo_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_algo_state_from_raw(*, f, v, l, atom_types, node_index, edge_node_index, t_graph, t_nodes) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def clean_prediction(state: Algorithm19State, *, variant: str = 'minus', coordinate_score_mode: str = 'direct') -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=variant,
        coordinate_score_mode=coordinate_score_mode,
    )


def sample_gt_noisy_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state


def sample_facit_kldm_em(case: GraphCase, *, n_steps: int = 1000, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 50000 * int(case.graph_id) + int(n_steps))
    f_t, v_t, l_t, a_t = runner.model.sample_CSP_algorithm3_facit(
        n_steps=int(n_steps),
        batch=batch_view,
        t_start=1.0,
        t_final=1.0e-6,
    )
    return {
        'f': f_t.detach().clone(),
        'v': v_t.detach().clone(),
        'l': l_t.detach().clone().reshape(-1),
        'a': a_t.detach().clone(),
        'batch_view': batch_view,
    }


def sample_template_q(case: GraphCase, payload, *, mode: str, seed: int = SAMPLE_SEED, dtype=None, device=None):
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    dtype = case.gt_frac.dtype if dtype is None else dtype
    device = case.gt_frac.device if device is None else device
    if chart.total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'crystalformer_oracle', 'oracle_structure', 'oracle'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'random_template', 'random'}:
        set_seed(int(seed) + 30000 * int(case.graph_id))
        return torch.rand((int(chart.total_dof),), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q mode {mode!r}')


def build_template_f0_model(case: GraphCase, payload, *, q0: torch.Tensor):
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q0.reshape(-1), 1.0) if q0.numel() else q0.reshape(-1)
    z_payload = chart.expand_q(q_eval, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
    return f0_model.detach().clone(), z_payload.detach().clone()


def sample_template_noisy_state(case: GraphCase, *, init_mode: str, t_value: float, seed: int = SAMPLE_SEED):
    payload = build_oracle_payload(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, z0_payload = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    set_seed(int(seed) + 70000 * int(case.graph_id) + int(round(1000.0 * float(t_value))) + (0 if init_mode == 'crystalformer_oracle' else 1))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state, payload, q0, z0_payload


def q_distance(a: torch.Tensor, b: torch.Tensor) -> float:
    if a.numel() == 0 and b.numel() == 0:
        return 0.0
    diff = wrapdiff(a.reshape(-1), b.reshape(-1))
    return float(torch.sqrt(diff.square().mean()).detach().item())


def composition_to_formula_string(composition: dict[int, int]) -> str:
    counts = [int(v) for v in composition.values() if int(v) > 0]
    gcd_value = 0
    for value in counts:
        gcd_value = int(value) if gcd_value == 0 else math.gcd(int(gcd_value), int(value))
    gcd_value = gcd_value or 1
    parts = []
    for atomic_number in sorted(int(k) for k in composition):
        count = int(composition[atomic_number])
        if count <= 0:
            continue
        reduced = count // gcd_value
        symbol = Element.from_Z(int(atomic_number)).symbol
        parts.append(symbol if reduced == 1 else f'{symbol}{reduced}')
    return ''.join(parts)


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def config21(*, beta: float, alpha: float, q_opt_steps: int = Q_OPT_STEPS, post_renoise_acceptance: bool = True, debug_prints: bool = False, cf_grad_max_dims: int | None = None, cf_value_only_after_renoise: bool = False) -> Algorithm21Config:
    return Algorithm21Config(
        beta=float(beta),
        alpha=float(alpha),
        q_opt_steps=int(q_opt_steps),
        post_renoise_acceptance=bool(post_renoise_acceptance),
        debug_prints=bool(debug_prints),
        cf_grad_max_dims=cf_grad_max_dims,
        cf_value_only_after_renoise=bool(cf_value_only_after_renoise),
    )


def fit_clean_q(clean_f: torch.Tensor, *, case: GraphCase, payload, config: Algorithm21Config, cf_like=None, formula: str | None = None, q_init: torch.Tensor | None = None):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    return q_only_clean_cf_fit(
        z_payload=z_payload,
        payload=payload,
        t_nodes=torch.full((int(clean_f.shape[0]),), 0.3, device=clean_f.device, dtype=clean_f.dtype),
        lattice_feature=case.gt_l_feature,
        formula=formula,
        config=config,
        cf_likelihood=cf_like,
        q_init=q_init,
    )


loaded_graphs= [2] sg= [4]


## Notebook Scope

This notebook focuses on the new Algorithm23-specific questions rather than replaying the full Algorithm22 audit ladder. The stable setup, PyXtal candidate generation, and sample-evaluation metrics are reused, but the tests are all about whether local CF-MCMC improves q safely.


In [5]:
audit_rows = [
    {'spec_version': 'old_algorithm22_markdown', 'area': 'A-F', 'status': 'audited', 'note': 'Used as baseline for earlier backend/notebook draft.'},
    {'spec_version': 'current_algorithm22_update', 'area': 'A-F', 'status': 'audited', 'note': 'This notebook tracks the current requested tests and implementation status.'},
]
source_rows = [
    {'source': 'src/PPR/ppr-main/ppr/projection.py', 'verified_point': 'CPS/PPR project-clean-then-continue principle.'},
    {'source': 'src/PPR/ppr-main/ppr/diffusion/samplers.py', 'verified_point': 'Projection lives inside the reverse-step flow after prediction.'},
    {'source': 'src/kldmPlus/symmetry/pcs_projection.py', 'verified_point': 'PyXtal-backed exact-composition template states already exist.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/sample.py', 'verified_point': 'CrystalFormer reduced-variable sampling remains expensive and is treated carefully.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/loss.py', 'verified_point': 'CrystalFormer coordinate NLL is used as a score, not the hard symmetry constraint.'},
]
symbol_rows = []
for name in [
    'algorithm22_projection_schedule', 'algorithm22_piecewise_alpha', 'algorithm22A_oracle_template_kldm_cps',
    'algorithm22B_ranked_kldm_cps', 'generate_candidates_22', 'sample_template_q_proposals_22',
    'expand_template_to_model_order_22', 'rank_projected_candidates_with_rule_22', 'safety_ok_22',
]:
    symbol_rows.append({'backend_symbol': name, 'available': bool(name in globals())})
display(pd.DataFrame(audit_rows))
display(pd.DataFrame(source_rows))
display(pd.DataFrame(symbol_rows))


,spec_version,area,status,note
0,old_algorithm22_markdown,A-F,audited,Used as baseline for earlier backend/notebook ...
1,current_algorithm22_update,A-F,audited,This notebook tracks the current requested tes...


,source,verified_point
0,src/PPR/ppr-main/ppr/projection.py,CPS/PPR project-clean-then-continue principle.
1,src/PPR/ppr-main/ppr/diffusion/samplers.py,Projection lives inside the reverse-step flow ...
2,src/kldmPlus/symmetry/pcs_projection.py,PyXtal-backed exact-composition template state...
3,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer reduced-variable sampling remain...
4,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer coordinate NLL is used as a scor...


,backend_symbol,available
0,algorithm22_projection_schedule,True
1,algorithm22_piecewise_alpha,True
2,algorithm22A_oracle_template_kldm_cps,True
3,algorithm22B_ranked_kldm_cps,True
4,generate_candidates_22,True
5,sample_template_q_proposals_22,True
6,expand_template_to_model_order_22,True
7,rank_projected_candidates_with_rule_22,True
8,safety_ok_22,True


## Runtime Preflight

CrystalFormer is still kept out of the notebook kernel during heavy work. The notebook uses subprocess smoke checks plus subprocess scoring helpers, then runs Algorithm23 logic around those safe energy evaluations.


In [6]:

rows = []
CF_READY = False
CF_COORDINATE_ONLY = np.nan
CF_RUNTIME_MODE = 'none'
CF_SMOKE_ERROR_TYPE = None
CF_SMOKE_ERROR_MESSAGE = None
CF_KERNEL_LIKE = None
CF_IMPORTS = {}
CF_WORKER_PROC = None
CF_WORKER_STARTUP_TIMEOUT = float(os.environ.get('KLDM_ALGO23_CF_STARTUP_TIMEOUT', '90'))
CF_LAST_ERROR_TYPE = None
CF_LAST_ERROR_MESSAGE = None
notebook_log('[algo23.preflight] cf checkpoint load start')


def _try_import_flag(module_name: str) -> bool:
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False


def _clear_cf_error() -> None:
    global CF_LAST_ERROR_TYPE, CF_LAST_ERROR_MESSAGE
    CF_LAST_ERROR_TYPE = None
    CF_LAST_ERROR_MESSAGE = None


def _record_cf_error(exc: Exception) -> None:
    global CF_LAST_ERROR_TYPE, CF_LAST_ERROR_MESSAGE, CF_SMOKE_ERROR_TYPE, CF_SMOKE_ERROR_MESSAGE
    CF_LAST_ERROR_TYPE = type(exc).__name__
    CF_LAST_ERROR_MESSAGE = ''.join(traceback.format_exception_only(type(exc), exc)).strip()
    CF_SMOKE_ERROR_TYPE = CF_LAST_ERROR_TYPE
    CF_SMOKE_ERROR_MESSAGE = CF_LAST_ERROR_MESSAGE


def _start_cf_worker_algo23():
    global CF_WORKER_PROC, CF_READY, CF_COORDINATE_ONLY, CF_RUNTIME_MODE, CF_SMOKE_ERROR_TYPE, CF_SMOKE_ERROR_MESSAGE
    if CF_WORKER_PROC is not None and getattr(CF_WORKER_PROC, 'poll', lambda: None)() is None:
        return CF_WORKER_PROC
    cmd = [
        sys.executable,
        str(ROOT / 'scripts' / 'algorithm23_cf_worker.py'),
        '--repo-root', str(ROOT),
        '--checkpoint', str(CF_CHECKPOINT),
    ]
    notebook_log('[algo23.worker] launching persistent CrystalFormer worker on first use')
    proc = subprocess.Popen(
        cmd,
        cwd=str(ROOT),
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        start_new_session=True,
    )
    msg = None
    raw_lines = []
    start_time = time.time()
    if proc.stdout is None:
        raise RuntimeError('CrystalFormer worker stdout unavailable')
    for _ in range(500):
        if (time.time() - start_time) > CF_WORKER_STARTUP_TIMEOUT:
            proc.kill()
            raise TimeoutError(f'CrystalFormer worker startup exceeded {CF_WORKER_STARTUP_TIMEOUT:.1f}s')
        line = proc.stdout.readline()
        if not line:
            break
        raw_lines.append(line.rstrip())
        stripped = line.strip()
        if not stripped:
            continue
        try:
            candidate = jsonlib.loads(stripped)
        except Exception:
            continue
        msg = candidate
        break
    if msg is None:
        stderr_tail = proc.stderr.read()[-1000:] if proc.stderr is not None else ''
        raise RuntimeError(f'CrystalFormer worker produced no JSON startup message. raw_lines={raw_lines[-5:]} stderr_tail={stderr_tail}')
    if not msg.get('ok'):
        raise RuntimeError(msg.get('error_message') or 'CrystalFormer worker failed to start')
    CF_WORKER_PROC = proc
    CF_READY = True
    CF_COORDINATE_ONLY = bool(msg.get('coordinate_only', True))
    CF_RUNTIME_MODE = 'subprocess_worker'
    _clear_cf_error()
    CF_SMOKE_ERROR_TYPE = None
    CF_SMOKE_ERROR_MESSAGE = None
    notebook_log('[algo23.worker] persistent CrystalFormer worker ready')
    return proc


def _cf_worker_request(request: dict[str, Any]) -> dict[str, Any]:
    proc = _start_cf_worker_algo23()
    if proc.stdin is None or proc.stdout is None:
        raise RuntimeError('CrystalFormer worker stdio unavailable')
    proc.stdin.write(jsonlib.dumps(request) + '\n')
    proc.stdin.flush()
    raw_lines = []
    msg = None
    for _ in range(500):
        line = proc.stdout.readline()
        if not line:
            break
        raw_lines.append(line.rstrip())
        stripped = line.strip()
        if not stripped:
            continue
        try:
            candidate = jsonlib.loads(stripped)
        except Exception:
            continue
        msg = candidate
        break
    if msg is None:
        stderr_tail = proc.stderr.read()[-1000:] if proc.stderr is not None else ''
        raise RuntimeError(f'CrystalFormer worker returned no JSON response. raw_lines={raw_lines[-5:]} stderr_tail={stderr_tail}')
    if not msg.get('ok'):
        raise RuntimeError(msg.get('error_message') or 'CrystalFormer worker request failed')
    return msg


try:
    ckpt_exists = Path(CF_CHECKPOINT).exists()
    for module_name in ('jax', 'haiku', 'optax'):
        CF_IMPORTS[module_name] = bool(_try_import_flag(module_name))

    runtime_preference = str(os.environ.get('KLDM_ALGO23_CF_RUNTIME_MODE', 'subprocess_primary')).strip().lower()
    if ckpt_exists and runtime_preference == 'kernel_primary':
        try:
            notebook_log('[algo23.preflight] trying in-kernel CrystalFormer runtime (opt-in)')
            CF_KERNEL_LIKE = CrystalFormerLikelihood(checkpoint_path=str(CF_CHECKPOINT))
            CF_READY = True
            CF_COORDINATE_ONLY = bool(CF_KERNEL_LIKE.coordinate_only)
            CF_RUNTIME_MODE = 'in_kernel_single_runtime'
            notebook_log('[algo23.preflight] in-kernel CrystalFormer runtime ready')
        except Exception as exc:
            CF_SMOKE_ERROR_TYPE = type(exc).__name__
            CF_SMOKE_ERROR_MESSAGE = ''.join(traceback.format_exception_only(type(exc), exc)).strip()
            notebook_log(f'[algo23.preflight] in-kernel runtime failed: {CF_SMOKE_ERROR_TYPE}: {CF_SMOKE_ERROR_MESSAGE}')
    elif ckpt_exists and runtime_preference == 'subprocess_primary':
        # Deferred worker startup keeps the notebook responsive; the real runtime
        # is launched on the first likelihood query.
        CF_READY = True
        CF_COORDINATE_ONLY = True
        CF_RUNTIME_MODE = 'deferred_subprocess_worker'
        notebook_log('[algo23.preflight] deferring CrystalFormer worker startup until first CF query')

    rows.append({
        'test': 'algorithm22_cf_checkpoint_load',
        'checkpoint_path': CF_CHECKPOINT,
        'checkpoint_exists': bool(ckpt_exists),
        'wrapper_ready': bool(CF_READY),
        'coordinate_only': CF_COORDINATE_ONLY,
        'runtime_mode': CF_RUNTIME_MODE,
        'python_executable': sys.executable,
        'jax_importable': bool(CF_IMPORTS.get('jax')),
        'haiku_importable': bool(CF_IMPORTS.get('haiku')),
        'optax_importable': bool(CF_IMPORTS.get('optax')),
        'smoke_error_type': CF_SMOKE_ERROR_TYPE,
        'smoke_error_message': CF_SMOKE_ERROR_MESSAGE,
        'PASS': bool(CF_READY),
        'status': 'INFO' if CF_READY else ('MISSING_CHECKPOINT' if not ckpt_exists else 'CF_RUNTIME_FAILED'),
    })
except Exception as exc:
    rows.append(error_row('algorithm22_cf_checkpoint_load', 'na', exc, 'cf_checkpoint_load', checkpoint_path=CF_CHECKPOINT))
safe_display_sorted(dataframe_result('algorithm22_cf_checkpoint_load', rows), ['checkpoint_path'])


[algo23.preflight] cf checkpoint load start
[algo23.preflight] deferring CrystalFormer worker startup until first CF query


,test,checkpoint_path,checkpoint_exists,wrapper_ready,coordinate_only,runtime_mode,python_executable,jax_importable,haiku_importable,optax_importable,smoke_error_type,smoke_error_message,PASS,status
0,algorithm22_cf_checkpoint_load,/workspace/notebooks/epoch_046000.pkl,True,True,True,deferred_subprocess_worker,/workspace/.venv/bin/python,True,True,True,None,None,True,INFO


In [7]:

ALGO22_CFG = Algorithm22Config(
    schedule=algo22_backend.Algorithm22ScheduleConfig(
        n_pc_steps=800,
        projection_interval=50,
        p_start=0.625,
        alpha_max=0.25,
    ),
    top_branches=1,
    state_return_mode='preserve_velocity_shift',
    lattice_projection=False,
    post_acceptance=True,
)
ALGO22_SCHEDULE = algorithm22_projection_schedule(schedule=ALGO22_CFG.schedule)
ALGO22_PROJECTION_STEPS = [int(pt.step) for pt in ALGO22_SCHEDULE if pt.project]
ALGO22_LOCAL_STEPS = [600, 750]
RUN_ALGO22_CF_HEAVY = str(os.environ.get('RUN_ALGO22_CF_HEAVY', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
ALGO22_PHASEA_SOURCES = ['baseline', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])
ALGO22_PHASEE_SOURCES = ['baseline', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score', 'pyxtal_cf_q_augmented'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])


def algo22_step_to_progress(step: int, n_pc_steps: int = 800) -> float:
    return float(step) / float(max(1, n_pc_steps - 1))


def algo22_step_to_t(step: int, n_pc_steps: int = 800) -> float:
    return float(max(1.0e-6, 1.0 - algo22_step_to_progress(step, n_pc_steps=n_pc_steps)))


def oracle_signature(case: GraphCase):
    payload = build_oracle_payload(case)
    return tuple(sorted((int(z), str(label)) for z, label in zip(payload.anchor_atomic_numbers.tolist(), payload.wyckoff_letters)))


def template_signature(template) -> tuple:
    return flatten_site_signature(template)


def candidate_rmse_to_gt(case: GraphCase, frac_coords_model_order: torch.Tensor) -> float:
    return float(evaluate_arrays(case, frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)['frac_rmse'])


def safe_metric_float(value, default: float = float('nan')) -> float:
    if value is None:
        return float(default)
    try:
        out = float(value)
    except Exception:
        return float(default)
    return out if np.isfinite(out) else float(default) if not np.isnan(default) else out


def safe_metric_bool(value) -> bool:
    return bool(False if value is None else value)


def eval_metric_bundle(ev: dict[str, Any]) -> dict[str, Any]:
    return {
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
        'composition_match': safe_metric_bool(ev.get('composition_match')),
        'requested_space_group_match': safe_metric_bool(ev.get('requested_space_group_match')),
        'standardized_frac_rmse': safe_metric_float(ev.get('standardized_frac_rmse')),
    }


def topk_min(values, k: int):
    vals = [float(v) for v in values if np.isfinite(float(v))]
    if not vals:
        return float('nan')
    return float(min(sorted(vals)[: max(1, int(k))]))


ALGO22_CF_SCORE_CANDIDATE_LIMIT = int(os.environ.get('ALGO22_CF_SCORE_CANDIDATE_LIMIT', '4'))
ALGO22_CF_AUG_BASE_LIMIT = int(os.environ.get('ALGO22_CF_AUG_BASE_LIMIT', '1'))


def cf_nll_eval_batch(*, items, label: str) -> list[float]:
    if not CF_READY or not items:
        return [float('nan')] * len(items)
    tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
    in_path = tmp_dir / f'{safe_label}_batch.pkl'
    out_path = tmp_dir / f'{safe_label}_batch.json'
    packed = []
    for item in items:
        packed.append({
            'symmetry_payload': item['payload'],
            'q': np.asarray(torch.as_tensor(item['q']).detach().cpu(), dtype=float),
            'lattice_feature': np.asarray(torch.as_tensor(item['lattice_feature']).detach().cpu(), dtype=float),
            'formula': item.get('formula'),
        })
    with in_path.open('wb') as handle:
        pickle.dump({
            'repo_root': str(ROOT),
            'checkpoint_path': str(CF_CHECKPOINT),
            'items': packed,
        }, handle)
    cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_score_batch_once.py'), '--input', str(in_path), '--output', str(out_path)]
    notebook_log(f'[cf-eval-batch] subprocess start {label} n={len(items)}')
    completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
    result = {}
    if out_path.exists():
        with out_path.open('r', encoding='utf-8') as handle:
            result = jsonlib.load(handle)
    if completed.returncode != 0 or not result.get('ok'):
        notebook_log(f"[cf-eval-batch] ERROR {label} returncode={completed.returncode} stderr_tail={completed.stderr[-500:]}")
        raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer batch subprocess failed')
    values = [float(v) for v in result.get('values', [])]
    if len(values) != len(items):
        raise RuntimeError(f'CrystalFormer batch returned {len(values)} values for {len(items)} items')
    return values


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def q_strategy_label(strategy: str) -> str:
    mapping = {
        'pcs_anchor': 'pcs_anchor',
        'uniform': 'uniform',
        'sobol': 'sobol',
        'special-position-biased': 'special',
        'local': 'local',
        'small_local_perturbations': 'local',
    }
    return mapping.get(str(strategy), str(strategy))


def candidate_cf_stats(candidates, *, cf_mode: str):
    cf_vals = [float(c.cf_nll) for c in candidates if np.isfinite(float(c.cf_nll))]
    sampled = sum(1 for c in candidates if 'cf_sample_index' in (c.metadata or {}))
    return {
        'cf_active': bool(CF_READY and cf_mode != 'none'),
        'cf_mode': str(cf_mode),
        'cf_nll_finite': bool(len(cf_vals) > 0),
        'num_cf_q_samples': int(sampled),
        'num_cf_scored': int(len(cf_vals)),
        'cf_nll_min': float(np.min(cf_vals)) if cf_vals else float('nan'),
        'cf_nll_max': float(np.max(cf_vals)) if cf_vals else float('nan'),
        'cf_nll_has_nan': bool(any(not np.isfinite(float(c.cf_nll)) for c in candidates)),
    }


def make_algo22_cfg(*, p_start: float | None = None, q_sampling_strategies=None, num_q_per_template: int | None = None, top_branches: int | None = None, cf_sample_k: int | None = None, source_mode: str | None = None, state_return_mode: str | None = None):
    cfg = ALGO22_CFG
    if p_start is not None:
        cfg = replace(cfg, schedule=replace(cfg.schedule, p_start=float(p_start)))
    if q_sampling_strategies is not None:
        cfg = replace(cfg, pyxtal_q_sampling_strategies=tuple(q_sampling_strategies))
    if num_q_per_template is not None:
        cfg = replace(cfg, pyxtal_q_samples_per_template=int(num_q_per_template))
    if top_branches is not None:
        cfg = replace(cfg, top_branches=int(top_branches))
    if cf_sample_k is not None:
        cfg = replace(cfg, cf_sample_k=int(cf_sample_k))
    if source_mode is not None:
        cfg = replace(cfg, crystalformer_template_mode=str(source_mode))
    if state_return_mode is not None:
        cfg = replace(cfg, state_return_mode=str(state_return_mode))
    return cfg


def sample_q_candidates_eval(*, payload, lattice_feature, formula, K: int, seed: int, top_p: float = 1.0, temperature: float = 1.0):
    if not CF_READY:
        return [], []
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    if use_subprocess:
        tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
        tmp_dir.mkdir(parents=True, exist_ok=True)
        in_path = tmp_dir / f'algorithm22_sample_q_K{int(K)}_seed{int(seed)}.pkl'
        out_path = tmp_dir / f'algorithm22_sample_q_K{int(K)}_seed{int(seed)}.json'
        with in_path.open('wb') as handle:
            pickle.dump({
                'repo_root': str(ROOT),
                'checkpoint_path': str(CF_CHECKPOINT),
                'symmetry_payload': payload,
                'lattice_feature': np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float),
                'formula': formula,
                'K': int(K),
                'top_p': float(top_p),
                'temperature': float(temperature),
                'seed': int(seed),
            }, handle)
        cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_sample_q_once.py'), '--input', str(in_path), '--output', str(out_path)]
        completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
        result = {}
        if out_path.exists():
            with out_path.open('r', encoding='utf-8') as handle:
                result = jsonlib.load(handle)
        if completed.returncode != 0 or not result.get('ok'):
            raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer q sampling subprocess failed')
        q_candidates = [torch.as_tensor(q_i, dtype=torch.as_tensor(lattice_feature).dtype, device=torch.as_tensor(lattice_feature).device) for q_i in result['q_candidates']]
        cf_nll = [float(v) for v in result.get('cf_nll', [])]
        return q_candidates, cf_nll
    return sample_q_from_crystalformer(
        payload=payload,
        lattice_feature=torch.as_tensor(lattice_feature),
        formula=formula,
        cf_likelihood=None,
        K=int(K),
        top_p=float(top_p),
        temperature=float(temperature),
        seed=int(seed),
    )


def generate_source_candidates(case: GraphCase, state, f0_hat: torch.Tensor, *, source: str, cfg):
    formula = cf_formula_for_case(case)
    cf_mode = 'none'
    if source in {'pyxtal_cf_score', 'pyxtal_cf_q_augmented'} and not RUN_ALGO22_CF_HEAVY:
        return tuple(), {
            'cf_active': bool(CF_READY),
            'cf_mode': 'disabled',
            'cf_nll_finite': False,
            'num_cf_q_samples': 0,
            'num_cf_scored': 0,
            'cf_nll_min': float('nan'),
            'cf_nll_max': float('nan'),
            'cf_nll_has_nan': False,
        }
    if source == 'pyxtal_only':
        candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src={source}',
        )
        candidates = tuple(candidates[: max(1, int(ALGO22_PYXTAL_CANDIDATE_LIMIT))])
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    if source == 'pyxtal_cf_score':
        cf_mode = 'score_only'
        base_candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src=pyxtal_only_for_cf_score',
        )
        base_candidates = tuple(base_candidates[: max(1, int(ALGO22_CF_SCORE_CANDIDATE_LIMIT))])
        nll_values = cf_nll_eval_batch(items=[
            {'payload': cand.payload, 'q': cand.q_init, 'lattice_feature': cand.lattice_feature, 'formula': formula}
            for cand in base_candidates
        ], label=f'g{case.graph_id}_cfscore_batch') if base_candidates else []
        scored = []
        for cand, nll in zip(base_candidates, nll_values):
            scored.append(replace(cand, source='pyxtal_cf_score', cf_nll=float(nll), metadata={**(cand.metadata or {}), 'cf_scored': True}))
        candidates = tuple(scored)
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    if source == 'pyxtal_cf_q_augmented':
        cf_mode = 'q_augmented'
        base_candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src=pyxtal_only_for_cf_aug',
        )
        augmented = []
        for base_idx, cand in enumerate(base_candidates[: min(len(base_candidates), int(ALGO22_CF_AUG_BASE_LIMIT))]):
            augmented.append(replace(cand, source='pyxtal_cf_q_augmented', metadata={**(cand.metadata or {}), 'cf_aug_base': True}))
            q_candidates, cf_nll_values = sample_q_candidates_eval(
                payload=cand.payload,
                lattice_feature=cand.lattice_feature,
                formula=formula,
                K=int(cfg.cf_sample_k),
                top_p=float(cfg.cf_top_p),
                temperature=float(cfg.cf_temperature),
                seed=int(cfg.cf_sampler_seed) + int(case.graph_id) * 1000 + int(base_idx),
            )
            for sample_idx, (q_i, nll) in enumerate(zip(q_candidates, cf_nll_values)):
                try:
                    aug = materialize_candidate_from_template_22(
                        template=cand.template,
                        q=q_i.reshape(-1),
                        state=state,
                        lattice_matrix=cand.lattice_matrix,
                        lattice_feature=cand.lattice_feature,
                        source='pyxtal_cf_q_augmented',
                        metadata={**(cand.metadata or {}), 'cf_sample_index': int(sample_idx), 'cf_aug_base_index': int(base_idx)},
                        cf_nll=float(nll),
                        spacegroup=case.requested_sg,
                    )
                    augmented.append(aug)
                except Exception:
                    continue
        candidates = tuple(augmented)
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    raise ValueError(f'Unsupported source: {source}')


def fit_candidate_to_clean(case: GraphCase, state, f0_hat: torch.Tensor, candidate, cfg):
    fitted = algo22_backend.fit_q_to_template(
        candidate=candidate,
        target_f0=f0_hat,
        target_atomic_numbers=state.atom_types,
        q_init=candidate.q_init,
        lambda_init=0.0,
        q_opt_steps=int(cfg.q_opt_steps),
        q_lr=float(cfg.q_lr),
        grad_clip=float(cfg.grad_clip),
    )
    return fitted


ALGO22_LOCAL_SOURCES = ['no_correction', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score', 'pyxtal_cf_q_augmented'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])
ALGO22_PYXTAL_CANDIDATE_LIMIT = int(os.environ.get('ALGO22_PYXTAL_CANDIDATE_LIMIT', '8'))
REPLAY_CACHE = {}


def fit_oracle_payload_from_q_init(case: GraphCase, state, f0_hat: torch.Tensor, payload, *, q_init, cfg, init_label: str):
    q_gt = torch.as_tensor(algo22_backend._get_wyckoff_dof_chart(payload).q_ref, device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    z_hat = model_to_payload(f_model=f0_hat, payload=payload)
    q_init_tensor = None if q_init is None else torch.as_tensor(q_init, device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    fit = fit_q_to_clean_prediction(
        z_hat=z_hat,
        payload=payload,
        t_nodes=state.t_nodes,
        lattice_feature=state.l,
        q_init=q_init_tensor,
        q_init_mode='random' if q_init_tensor is None else 'oracle_structure',
        steps=int(cfg.q_opt_steps),
        lr=float(cfg.q_lr),
        grad_clip=float(cfg.grad_clip),
    )
    f_proj = payload_to_model(z_payload=fit.z_proj_payload, payload=payload)
    ev = evaluate_arrays(case, f_proj, case.gt_l_feature, case.atomic_numbers)
    q_init_dist = float('nan') if q_init_tensor is None or q_init_tensor.numel() != q_gt.numel() else float(torch.sqrt(cf_backend.torus_mse(q_init_tensor.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item())
    q_fit_dist = float('nan') if fit.q_star.numel() != q_gt.numel() else float(torch.sqrt(cf_backend.torus_mse(fit.q_star.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item())
    return {
        'init_label': str(init_label),
        'q_init_distance_to_GT': q_init_dist,
        'q_fit_distance_to_GT': q_fit_dist,
        'witness_after': float(fit.witness_sin),
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
    }


def ranked_summary(case: GraphCase, ranked) -> dict[str, Any]:
    if not ranked:
        return {
            'top1_frac_rmse': float('nan'),
            'top1_rmse': float('nan'),
            'top1_match': False,
            'top1_valid': False,
            'top3_min_frac_rmse': float('nan'),
        }
    top1_eval = evaluate_arrays(case, ranked[0].frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)
    top3_min = min([candidate_rmse_to_gt(case, item.frac_coords_model_order) for item in ranked[:3]], default=float('nan'))
    return {
        'top1_frac_rmse': float(top1_eval['frac_rmse']),
        'top1_rmse': float(top1_eval['rmse']),
        'top1_match': bool(top1_eval['match']),
        'top1_valid': bool(top1_eval['valid']),
        'top3_min_frac_rmse': float(top3_min),
    }


def replay_source(case: GraphCase, *, source: str, cfg, alpha_override=None):
    cache_key = (int(case.graph_id), str(source), repr(cfg), None if alpha_override is None else float(alpha_override))
    if cache_key in REPLAY_CACHE:
        return REPLAY_CACHE[cache_key]
    rows = []
    baseline_rows = []
    payload = build_oracle_payload(case)
    for step in ALGO22_PROJECTION_STEPS:
        state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(step))
        f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=cfg)
        base_eval = evaluate_arrays(case, f0_hat, case.gt_l_feature, case.atomic_numbers)
        base_witness = float(oracle_template_witness_22(state=state, model=runner.model, payload=payload, config=cfg))
        baseline_rows.append({'step': step, 'frac_rmse': safe_metric_float(base_eval.get('frac_rmse')), 'rmse': safe_metric_float(base_eval.get('rmse')), 'match': safe_metric_bool(base_eval.get('match')), 'valid': safe_metric_bool(base_eval.get('valid')), 'witness': base_witness})
        alpha = float(algorithm22_piecewise_alpha(algo22_step_to_progress(step)) if alpha_override is None else alpha_override)
        if source == 'baseline':
            rows.append({**baseline_rows[-1], 'accepted': True, 'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0})
            continue
        if source == 'oracle_template':
            branch = oracle_template_projection_22(state=state, model=runner.model, payload=payload, alpha=alpha, beta=1.0, config=cfg)
            ev = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
            rows.append({'step': step, 'frac_rmse': safe_metric_float(ev.get('frac_rmse')), 'rmse': safe_metric_float(ev.get('rmse')), 'match': safe_metric_bool(ev.get('match')), 'valid': safe_metric_bool(ev.get('valid')), 'witness': float(branch.witness_after), 'accepted': bool(branch.accepted), 'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0})
            continue
        candidates, cf_stats = generate_source_candidates(case, state, f0_hat, source=source, cfg=cfg)
        ranked = rank_candidates_against_clean_estimate_22(f0_hat=f0_hat, state=state, candidates=candidates, config=cfg)
        if not ranked:
            rows.append({'step': step, 'frac_rmse': float('nan'), 'rmse': float('nan'), 'match': False, 'valid': False, 'witness': float('inf'), 'accepted': False, **cf_stats})
            continue
        branch = algo22_backend.branch_survival_step(state=state, model=runner.model, projected=ranked[0], alpha=alpha, beta=1.0, candidate_pool=tuple(candidates), config=cfg)
        ev = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
        rows.append({'step': step, 'frac_rmse': safe_metric_float(ev.get('frac_rmse')), 'rmse': safe_metric_float(ev.get('rmse')), 'match': safe_metric_bool(ev.get('match')), 'valid': safe_metric_bool(ev.get('valid')), 'witness': float(branch.witness_after), 'accepted': bool(branch.accepted), **cf_stats})
    baseline_frac = float(np.nanmean([r['frac_rmse'] for r in baseline_rows])) if baseline_rows else float('nan')
    baseline_rmse = float(np.nanmean([r['rmse'] for r in baseline_rows])) if baseline_rows else float('nan')
    baseline_witness = float(np.nanmean([r['witness'] for r in baseline_rows])) if baseline_rows else float('nan')
    finite_rows = [r for r in rows if np.isfinite(float(r['frac_rmse']))]
    best_row = min(finite_rows, key=lambda r: float(r['frac_rmse'])) if finite_rows else None
    out = {
        'rows': rows,
        'baseline_rows': baseline_rows,
        'frac_rmse': float(np.nanmean([r['frac_rmse'] for r in rows])) if rows else float('nan'),
        'rmse': float(np.nanmean([r['rmse'] for r in rows])) if rows else float('nan'),
        'match_rate': float(np.mean([bool(r['match']) for r in rows])) if rows else float('nan'),
        'valid_rate': float(np.mean([bool(r['valid']) for r in rows])) if rows else float('nan'),
        'group_witness': float(np.nanmean([r['witness'] for r in rows])) if rows else float('nan'),
        'accepted_fraction': float(np.mean([bool(r['accepted']) for r in rows])) if rows else float('nan'),
        'projection_count': int(len(rows)),
        'best_step': int(best_row['step']) if best_row is not None else -1,
        'best_step_frac_rmse': float(best_row['frac_rmse']) if best_row is not None else float('nan'),
        'best_step_rmse_to_GT': float(best_row['rmse']) if best_row is not None else float('nan'),
        'best_step_match': bool(best_row['match']) if best_row is not None else False,
        'best_step_valid': bool(best_row['valid']) if best_row is not None else False,
        'beats_baseline': bool(np.nanmean([r['frac_rmse'] for r in rows]) < baseline_frac) if rows and np.isfinite(baseline_frac) else False,
        'improves_witness': bool(np.nanmean([r['witness'] for r in rows]) < baseline_witness) if rows and np.isfinite(baseline_witness) else False,
        'improves_frac_rmse': bool(np.nanmean([r['frac_rmse'] for r in rows]) < baseline_frac) if rows and np.isfinite(baseline_frac) else False,
        'improves_rmse': bool(np.nanmean([r['rmse'] for r in rows]) < baseline_rmse) if rows and np.isfinite(baseline_rmse) else False,
    }
    REPLAY_CACHE[cache_key] = out
    return out


In [8]:
ALGO23_CFG = Algorithm23Config(
    algo22=ALGO22_CFG,
    trust_region=Algorithm23TrustRegionConfig(
        relative_slack=0.10,
        absolute_slack=1.0e-6,
        shift_cap=0.02,
    ),
    mcmc=Algorithm23MCMCConfig(
        steps=16,
        proposal_sigmas=(0.005, 0.01, 0.02),
        acceptance_mode='greedy',
        temperature=1.0,
        seed=SAMPLE_SEED,
    ),
)
ALGO23_ORACLE_STEPS = [600, 700, 750]
ALGO23_ALPHA_VALUES = [0.05, 0.10]
ALGO23_TOP_K_TEMPLATES = [1, 3]
ALGO23_SMOKE_PROPOSALS = 8
ALGO23_SEEDS = [SAMPLE_SEED, SAMPLE_SEED + 1, SAMPLE_SEED + 2]
ALGO23_STRICT_TOL = float(os.environ.get('ALGO23_STRICT_TOL', '1e-6'))
ALGO23_CF_EM_K = int(os.environ.get('ALGO23_CF_EM_K', '1000'))
ALGO23_CF_EM_EPSILONS = (0.005, 0.01, 0.02)
ALGO23_CF_EM_ANALYSIS_TOPK = int(os.environ.get('ALGO23_CF_EM_ANALYSIS_TOPK', '10'))
ALGO23_ORACLE_STRESS_STEP = 750
ALGO23_ORACLE_STRESS_CONFIGS = (
    {'label': 'greedy_small_tr10_s8', 'mcmc_steps': 8, 'proposal_sigmas': (0.005, 0.01, 0.02), 'trust_region_relative': 0.10, 'acceptance_mode': 'greedy', 'temperature': 1.0},
    {'label': 'greedy_small_tr10_s32', 'mcmc_steps': 32, 'proposal_sigmas': (0.005, 0.01, 0.02), 'trust_region_relative': 0.10, 'acceptance_mode': 'greedy', 'temperature': 1.0},
    {'label': 'greedy_wide_tr20_s32', 'mcmc_steps': 32, 'proposal_sigmas': (0.01, 0.02, 0.05), 'trust_region_relative': 0.20, 'acceptance_mode': 'greedy', 'temperature': 1.0},
    {'label': 'metro_wide_tr20_s32', 'mcmc_steps': 32, 'proposal_sigmas': (0.01, 0.02, 0.05), 'trust_region_relative': 0.20, 'acceptance_mode': 'metropolis', 'temperature': 1.0},
)
ALGO23_NONORACLE_STRESS_CONFIGS = (
    {'label': 'greedy_small_tr10_s8', 'mcmc_steps': 8, 'proposal_sigmas': (0.005, 0.01), 'trust_region_relative': 0.10, 'acceptance_mode': 'greedy', 'temperature': 1.0},
    {'label': 'greedy_wide_tr20_s16', 'mcmc_steps': 16, 'proposal_sigmas': (0.01, 0.02, 0.05), 'trust_region_relative': 0.20, 'acceptance_mode': 'greedy', 'temperature': 1.0},
)


def make_algo23_cfg(*, mcmc_steps: int | None = None, proposal_sigmas: tuple[float, ...] | None = None, trust_region_relative: float | None = None, acceptance_mode: str | None = None, temperature: float | None = None):
    return Algorithm23Config(
        algo22=ALGO22_CFG,
        trust_region=Algorithm23TrustRegionConfig(
            relative_slack=float(ALGO23_CFG.trust_region.relative_slack if trust_region_relative is None else trust_region_relative),
            absolute_slack=float(ALGO23_CFG.trust_region.absolute_slack),
            shift_cap=float(ALGO23_CFG.trust_region.shift_cap),
        ),
        mcmc=Algorithm23MCMCConfig(
            steps=int(ALGO23_CFG.mcmc.steps if mcmc_steps is None else mcmc_steps),
            proposal_sigmas=tuple(ALGO23_CFG.mcmc.proposal_sigmas if proposal_sigmas is None else proposal_sigmas),
            acceptance_mode=str(ALGO23_CFG.mcmc.acceptance_mode if acceptance_mode is None else acceptance_mode),
            temperature=float(ALGO23_CFG.mcmc.temperature if temperature is None else temperature),
            seed=int(SAMPLE_SEED),
        ),
    )


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def cf_nll_eval_batch_algo23(*, items, label: str) -> list[float]:
    if not CF_READY or not items:
        return [float('nan')] * len(items)
    if str(CF_RUNTIME_MODE).startswith('in_kernel'):
        notebook_log(f"[cf-eval-batch-algo23] in-kernel start {label} n={len(items)}")
        cf_like = get_cf_like_algo23()
        values = []
        for item in items:
            values.append(float(cf_like.nll_q(payload=item['payload'], q=item['q'], lattice_feature=item['lattice_feature'], formula=item.get('formula'))))
        notebook_log(f"[cf-eval-batch-algo23] in-kernel done {label} n={len(values)}")
        return [safe_metric_float(v) for v in values]
    if str(CF_RUNTIME_MODE) in {'subprocess_worker', 'deferred_subprocess_worker'}:
        tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
        tmp_dir.mkdir(parents=True, exist_ok=True)
        safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
        in_path = tmp_dir / f'{safe_label}_algo23_batch.pkl'
        out_path = tmp_dir / f'{safe_label}_algo23_batch.json'
        packed = []
        for item in items:
            packed.append({
                'symmetry_payload': item['payload'],
                'q': np.asarray(torch.as_tensor(item['q']).detach().cpu(), dtype=float),
                'lattice_feature': np.asarray(torch.as_tensor(item['lattice_feature']).detach().cpu(), dtype=float),
                'formula': item.get('formula'),
            })
        with in_path.open('wb') as handle:
            pickle.dump({'items': packed}, handle)
        try:
            _cf_worker_request({'cmd': 'score_batch', 'input': str(in_path), 'output': str(out_path)})
        except Exception as exc:
            _record_cf_error(exc)
            notebook_log(f"[cf-eval-batch-algo23] worker error {label}: {CF_LAST_ERROR_TYPE}: {CF_LAST_ERROR_MESSAGE}")
            return [float('nan')] * len(items)
        result = jsonlib.loads(out_path.read_text(encoding='utf-8'))
        _clear_cf_error()
        values = [safe_metric_float(v) for v in result.get('values', [])]
        notebook_log(f"[cf-eval-batch-algo23] worker done {label} n={len(values)}")
        return values
    return [float('nan')] * len(items)


def cf_energy_for_algo23(candidate, q: torch.Tensor, lattice_feature: torch.Tensor, *, case: GraphCase, label: str) -> float:
    if not CF_READY:
        return float('nan')
    if str(CF_RUNTIME_MODE).startswith('in_kernel'):
        notebook_log(f"[cf-eval-algo23] in-kernel start {label}")
        cf_like = get_cf_like_algo23()
        value = float(cf_like.nll_q(payload=candidate.payload, q=q, lattice_feature=lattice_feature, formula=cf_formula_for_case(case)))
        notebook_log(f"[cf-eval-algo23] in-kernel done {label} value={value:.6g}")
        return value
    values = cf_nll_eval_batch_algo23(items=[{'payload': candidate.payload, 'q': q, 'lattice_feature': lattice_feature, 'formula': cf_formula_for_case(case)}], label=label)
    return float(values[0]) if values else float('nan')


def run_algo23_generate_proposals(case: GraphCase, state, f0_hat: torch.Tensor, candidate, cfg23, *, label: str):
    return algo23_backend.algorithm23_generate_cf_mcmc_q_proposals(
        candidate=candidate,
        target_f0=f0_hat,
        target_atomic_numbers=state.atom_types,
        lattice_feature=state.l,
        cf_energy_fn=lambda cand, q, lattice_feature: cf_energy_for_algo23(cand, q, lattice_feature, case=case, label=label),
        cf_formula=cf_formula_for_case(case),
        config=cfg23,
    )


def run_algo23_survival_selection(state, f0_hat: torch.Tensor, proposals_result, cfg23, *, alpha: float):
    return algo23_backend.algorithm23_select_by_kldm_survival(
        model=runner.model,
        state=state,
        f0_anchor=f0_hat,
        proposals=proposals_result.proposals,
        alpha=float(alpha),
        beta=1.0,
        config=cfg23,
        shift_weight=0.0,
    )


def proposal_best_by_cf(proposals_result):
    rows = [p for p in proposals_result.proposals if np.isfinite(float(p.cf_nll))]
    if not rows:
        return proposals_result.proposals[0]
    rows = sorted(rows, key=lambda p: (float(p.cf_nll), float(p.geometry_cost), int(p.step)))
    return rows[0]


def survival_eval_for_proposal(selection_result, proposal):
    for item in selection_result.evaluations:
        same_source = str(item.proposal.source) == str(proposal.source)
        same_step = int(item.proposal.step) == int(proposal.step)
        same_q = bool(torch.allclose(item.proposal.q, proposal.q, atol=1e-8, rtol=0.0))
        if same_source and same_step and same_q:
            return item
    return None


def evaluate_frac(case: GraphCase, frac: torch.Tensor) -> dict[str, Any]:
    return evaluate_arrays(case, frac, case.gt_l_feature, case.atomic_numbers)


def run_algo23_local_update(case: GraphCase, state, f0_hat: torch.Tensor, frac_target: torch.Tensor, alpha: float, cfg23):
    update = algorithm23_state_update(
        model=runner.model,
        state=state,
        f0_anchor=f0_hat,
        frac_target=frac_target,
        alpha=float(alpha),
        beta=1.0,
        config=cfg23,
    )
    f0_after = clean_fractional_estimate_22(state=update.state, model=runner.model, config=cfg23.algo22)
    return update, f0_after, evaluate_frac(case, f0_after)


def get_cf_like_algo23():
    global CF_KERNEL_LIKE, CF_READY, CF_COORDINATE_ONLY, CF_RUNTIME_MODE
    if CF_KERNEL_LIKE is not None:
        return CF_KERNEL_LIKE
    CF_KERNEL_LIKE = CrystalFormerLikelihood(checkpoint_path=str(CF_CHECKPOINT))
    CF_READY = True
    CF_COORDINATE_ONLY = bool(CF_KERNEL_LIKE.coordinate_only)
    CF_RUNTIME_MODE = 'in_kernel_single_runtime'
    return CF_KERNEL_LIKE


def recover_oracle_template_from_payload(case: GraphCase, payload):
    templates = extract_wyckoff_templates(
        space_group_number=int(payload.spacegroup),
        atomic_numbers=case.atomic_numbers,
        max_templates=64,
        quick=False,
    )
    signature = payload.site_signature
    template = None
    for candidate in templates:
        if flatten_site_signature(candidate) == signature:
            template = candidate
            break
    if template is None:
        raise RuntimeError(
            f"No oracle template matched payload signature for graph={case.graph_id} sg={int(payload.spacegroup)} signature={signature!r}."
        )
    q_ref = recover_template_free_vars_from_anchor_entries(template, payload.anchor_entries)
    return template, q_ref.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)


def oracle_template_candidate(case: GraphCase, state, payload, f0_hat: torch.Tensor):
    template, q_ref = recover_oracle_template_from_payload(case, payload)
    q_ref = q_ref.to(device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    lattice_matrix = decode_state_cell_matrix_22(state=state, lattice_transform=runner.lattice_transform)
    return materialize_candidate_from_template_22(
        template=template,
        q=q_ref,
        state=state,
        lattice_matrix=lattice_matrix,
        lattice_feature=state.l,
        source='oracle_template',
        metadata={'mode': 'oracle_template'},
        spacegroup=int(payload.spacegroup),
    )


def algorithm23_oracle_geom_fit(case: GraphCase, state, payload, f0_hat: torch.Tensor):
    candidate = oracle_template_candidate(case, state, payload, f0_hat)
    return fit_candidate_to_clean(case, state, f0_hat, candidate, ALGO22_CFG)


def bool_rate(values) -> float:
    vals = [bool(v) for v in values]
    return float(np.mean(vals)) if vals else float('nan')


def finite_min(values) -> float:
    vals = [float(v) for v in values if np.isfinite(float(v))]
    return float(min(vals)) if vals else float('nan')


def finite_mean(values) -> float:
    vals = [float(v) for v in values if np.isfinite(float(v))]
    return float(np.mean(vals)) if vals else float('nan')


def strict_improves(new_value, old_value, tol: float = ALGO23_STRICT_TOL) -> bool:
    new_f = safe_metric_float(new_value)
    old_f = safe_metric_float(old_value)
    return bool(np.isfinite(new_f) and np.isfinite(old_f) and new_f < old_f - float(tol))


def approximately_equal(new_value, old_value, tol: float = ALGO23_STRICT_TOL) -> bool:
    new_f = safe_metric_float(new_value)
    old_f = safe_metric_float(old_value)
    return bool(np.isfinite(new_f) and np.isfinite(old_f) and abs(new_f - old_f) <= float(tol))


def compare_metric(new_value, old_value, tol: float = ALGO23_STRICT_TOL) -> str:
    if strict_improves(new_value, old_value, tol=tol):
        return 'better'
    if strict_improves(old_value, new_value, tol=tol):
        return 'worse'
    if approximately_equal(new_value, old_value, tol=tol):
        return 'equal'
    return 'missing'


def accepted_count(refined) -> int:
    return int(sum(1 for step in refined.accepted_steps if bool(step.accepted)))


def chain_moved(refined) -> bool:
    return bool(accepted_count(refined) > 0)


## Test Group 0 — Smoke Tests

Decision goal: verify that CrystalFormer is active and that local q proposals inside the KLDM trust region produce nontrivial variation in CF NLL. If CF NLL is flat or almost all proposals leave the trust region, local MCMC will not help.


In [9]:

rows_0 = []
rows_02 = []
for case in GRAPH_CASES[:1]:
    payload = build_oracle_payload(case)
    state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(700))
    f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
    cand = oracle_template_candidate(case, state, payload, f0_hat)
    geom = fit_candidate_to_clean(case, state, f0_hat, cand, ALGO22_CFG)

    cf_values = []
    grad_available = False
    smoke_error_type = None
    smoke_error_message = None
    if CF_READY:
        try:
            cf_values.append(cf_energy_for_algo23(cand, geom.q_star, state.l, case=case, label=f'g{case.graph_id}_smoke_geom_1'))
            cf_values.append(cf_energy_for_algo23(cand, geom.q_star, state.l, case=case, label=f'g{case.graph_id}_smoke_geom_2'))
            if str(CF_RUNTIME_MODE).startswith('in_kernel'):
                try:
                    grad_available = bool(hasattr(get_cf_like_algo23(), 'grad_nll_q'))
                except Exception:
                    grad_available = False
        except Exception as exc:
            _record_cf_error(exc)
            smoke_error_type = CF_LAST_ERROR_TYPE
            smoke_error_message = CF_LAST_ERROR_MESSAGE
    finite_cf = [float(v) for v in cf_values if np.isfinite(float(v))]
    repeat_delta = abs(finite_cf[1] - finite_cf[0]) if len(finite_cf) >= 2 else float('nan')
    representation_stable = bool(len(finite_cf) >= 2 and repeat_delta <= 1.0e-6)
    rows_0.append({
        'test': 'algorithm23_smoke_cf_active',
        'graph': case.graph_id,
        'cf_active': bool(CF_READY),
        'cf_mode': str(CF_RUNTIME_MODE),
        'cf_nll_finite': bool(len(finite_cf) >= 1),
        'cf_nll_min': safe_metric_float(min(finite_cf) if finite_cf else float('nan')),
        'cf_nll_max': safe_metric_float(max(finite_cf) if finite_cf else float('nan')),
        'cf_repeat_delta_abs': safe_metric_float(repeat_delta),
        'representation_stable': bool(representation_stable),
        'cf_error_type': smoke_error_type or CF_LAST_ERROR_TYPE,
        'cf_error_message': smoke_error_message or CF_LAST_ERROR_MESSAGE,
        'cf_gradient_available': bool(grad_available),
        'cf_sampling_available': False,
        'PASS': bool(CF_READY and len(finite_cf) >= 1 and representation_stable),
        'decision': 'cf_ready_representation_stable' if (CF_READY and len(finite_cf) >= 1 and representation_stable) else ('cf_query_failed' if (smoke_error_type or CF_LAST_ERROR_TYPE) else 'cf_not_ready'),
        'status': 'INFO' if len(finite_cf) >= 1 else ('CF_QUERY_FAILED' if (smoke_error_type or CF_LAST_ERROR_TYPE) else ('BLOCKED_CF_RUNTIME' if not CF_READY else 'NO_FINITE_CF_NLL')),
    })

    proposal_costs = []
    proposal_items = []
    inside = 0
    geom_cap = algorithm23_geometry_cap(float(geom.witness), relative_slack=0.10, absolute_slack=1.0e-6)
    sigma_schedule = ([0.005, 0.01, 0.02, 0.05] * max(1, int(np.ceil(ALGO23_SMOKE_PROPOSALS / 4))))[:ALGO23_SMOKE_PROPOSALS]
    for sigma in sigma_schedule:
        q_prop = wrap01(geom.q_star + float(sigma) * torch.randn_like(geom.q_star))
        frac_prop = expand_template_to_model_order_22(template=cand.template, q=q_prop, lattice_matrix=cand.lattice_matrix, target_atomic_numbers=state.atom_types, reference_f0=f0_hat, spacegroup=int(cand.payload.spacegroup))
        cost = algorithm23_geometry_cost(frac_coords_model_order=frac_prop, target_f0=f0_hat)
        proposal_costs.append(cost)
        if cost <= geom_cap:
            inside += 1
            proposal_items.append({
                'payload': cand.payload,
                'q': q_prop,
                'lattice_feature': state.l,
                'formula': cf_formula_for_case(case),
            })
    proposal_nlls = cf_nll_eval_batch_algo23(items=proposal_items, label=f'g{case.graph_id}_smoke_props') if (CF_READY and proposal_items) else []
    finite_nlls = [float(v) for v in proposal_nlls if np.isfinite(float(v))]
    cf_varies = bool(len(finite_nlls) > 1 and float(np.std(finite_nlls)) > 1.0e-6)
    trust_ok = bool(inside >= max(1, len(proposal_costs) // 4))
    rows_02.append({
        'test': 'algorithm23_smoke_mcmc_proposal_validity',
        'graph': case.graph_id,
        'num_proposals': len(proposal_costs),
        'num_inside_trust_region': int(inside),
        'inside_trust_fraction': float(inside / max(1, len(proposal_costs))),
        'num_finite_cf_nll': int(len(finite_nlls)),
        'cf_nll_min': safe_metric_float(min(finite_nlls) if finite_nlls else float('nan')),
        'cf_nll_mean': safe_metric_float(np.mean(finite_nlls) if finite_nlls else float('nan')),
        'cf_nll_std': safe_metric_float(np.std(finite_nlls) if finite_nlls else float('nan')),
        'cf_error_type': CF_LAST_ERROR_TYPE,
        'cf_error_message': CF_LAST_ERROR_MESSAGE,
        'geometry_cost_min': safe_metric_float(min(proposal_costs) if proposal_costs else float('nan')),
        'geometry_cost_mean': safe_metric_float(np.mean(proposal_costs) if proposal_costs else float('nan')),
        'cf_varies_nontrivially': bool(cf_varies),
        'trust_region_viable': bool(trust_ok),
        'PASS': bool(CF_READY and trust_ok and cf_varies),
        'decision': 'mcmc_viable' if (CF_READY and trust_ok and cf_varies) else ('mcmc_cf_query_failed' if CF_LAST_ERROR_TYPE else 'mcmc_not_viable'),
        'status': 'INFO' if len(finite_nlls) >= 1 else ('CF_QUERY_FAILED' if CF_LAST_ERROR_TYPE else ('BLOCKED_CF_RUNTIME' if not CF_READY else 'NO_FINITE_CF_NLL')),
    })

safe_display_sorted(dataframe_result('algorithm23_smoke_cf_active', rows_0), ['graph'])
safe_display_sorted(dataframe_result('algorithm23_smoke_mcmc_proposal_validity', rows_02), ['graph'])


/workspace/src/kldmPlus/sample_evaluation/sample_evaluation.py:482: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  return torch.as_tensor(


[algo23.worker] launching persistent CrystalFormer worker on first use
[algo23.worker] persistent CrystalFormer worker ready
[cf-eval-batch-algo23] worker done g2_smoke_geom_1 n=1
[cf-eval-batch-algo23] worker done g2_smoke_geom_2 n=1
[cf-eval-batch-algo23] worker done g2_smoke_props n=4


,test,graph,cf_active,cf_mode,cf_nll_finite,cf_nll_min,cf_nll_max,cf_repeat_delta_abs,representation_stable,cf_error_type,cf_error_message,cf_gradient_available,cf_sampling_available,PASS,decision,status
0,algorithm23_smoke_cf_active,2,True,subprocess_worker,True,58.181782,58.181782,0.0,True,None,None,False,False,True,cf_ready_representation_stable,INFO


,test,graph,num_proposals,num_inside_trust_region,inside_trust_fraction,num_finite_cf_nll,cf_nll_min,cf_nll_mean,cf_nll_std,cf_error_type,cf_error_message,geometry_cost_min,geometry_cost_mean,cf_varies_nontrivially,trust_region_viable,PASS,decision,status
0,algorithm23_smoke_mcmc_proposal_validity,2,8,4,0.5,4,53.930244,59.723457,5.194684,None,None,0.000065,0.000697,True,True,True,mcmc_viable,INFO


# EM Test

Question: run a true 1000-step facitKLDM EM sampler, fit the oracle-template q implied by that final sample, and test whether GT q lies inside a small epsilon-neighborhood around that EM-anchored q basin.


In [ ]:
rows_em = []
rows_em_eps = []
for case in GRAPH_CASES[:1]:
    try:
        payload = build_oracle_payload(case)
        chart = algo22_backend._get_wyckoff_dof_chart(payload)
        q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
        em_result = sample_facit_kldm_em(case, n_steps=int(ALGO23_CF_EM_K), seed=int(SAMPLE_SEED))
        f_em = em_result['f']
        l_em = em_result['l']
        a_em = em_result['a']
        em_ev = evaluate_arrays(case, f_em, l_em, a_em)
        em_state = make_algo_state_from_raw(
            f=f_em,
            v=em_result['v'],
            l=l_em,
            atom_types=a_em,
            node_index=em_result['batch_view'].batch,
            edge_node_index=em_result['batch_view'].edge_node_index,
            t_graph=torch.full((1, 1), 1.0e-6, device=f_em.device, dtype=f_em.dtype),
            t_nodes=torch.full((f_em.shape[0], 1), 1.0e-6, device=f_em.device, dtype=f_em.dtype),
        )
        geom_from_em = algorithm23_oracle_geom_fit(case, em_state, payload, f_em)
        geom_em_ev = evaluate_frac(case, geom_from_em.frac_coords_model_order)
        q_em = geom_from_em.q_star.reshape(-1)
        q_em_dist = float(torch.sqrt(cf_backend.torus_mse(q_em, q_gt).clamp_min(0.0)).detach().item()) if q_em.numel() == q_gt.numel() else float('nan')
        neighborhood_rows = []
        for eps in ALGO23_CF_EM_EPSILONS:
            q_plus = wrap01(q_em + float(eps))
            q_minus = wrap01(q_em - float(eps))
            plus_dist = float(torch.sqrt(cf_backend.torus_mse(q_plus.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item()) if q_plus.numel() == q_gt.numel() else float('nan')
            minus_dist = float(torch.sqrt(cf_backend.torus_mse(q_minus.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item()) if q_minus.numel() == q_gt.numel() else float('nan')
            best_band = finite_min([q_em_dist, plus_dist, minus_dist])
            rows_em_eps.append({
                'test': 'algorithm23_em_test_cf_sampling_basin_epsilon',
                'graph': case.graph_id,
                'em_steps': int(ALGO23_CF_EM_K),
                'epsilon': float(eps),
                'q_em_distance_to_GT': safe_metric_float(q_em_dist),
                'q_em_plus_distance_to_GT': safe_metric_float(plus_dist),
                'q_em_minus_distance_to_GT': safe_metric_float(minus_dist),
                'best_distance_in_em_band': safe_metric_float(best_band),
                'gt_inside_em_band': bool(np.isfinite(best_band) and best_band <= float(eps)),
                'PASS': bool(np.isfinite(best_band) and best_band <= float(eps)),
                'status': 'INFO',
            })
            neighborhood_rows.append((float(eps), best_band))
        rows_em.append({
            'test': 'algorithm23_em_test_cf_sampling_basin',
            'graph': case.graph_id,
            'em_steps': int(ALGO23_CF_EM_K),
            'em_frac_rmse': safe_metric_float(em_ev.get('frac_rmse')),
            'em_rmse': safe_metric_float(em_ev.get('rmse')),
            'em_match': safe_metric_bool(em_ev.get('match')),
            'em_valid': safe_metric_bool(em_ev.get('valid')),
            'q_em_distance_to_GT': safe_metric_float(q_em_dist),
            'q_geom_from_em_frac_rmse': safe_metric_float(geom_em_ev.get('frac_rmse')),
            'q_geom_from_em_rmse': safe_metric_float(geom_em_ev.get('rmse')),
            'q_geom_from_em_match': safe_metric_bool(geom_em_ev.get('match')),
            'q_geom_from_em_valid': safe_metric_bool(geom_em_ev.get('valid')),
            'best_em_band_distance': finite_min([band for _, band in neighborhood_rows]),
            'best_em_band_epsilon': next((eps for eps, band in sorted(neighborhood_rows, key=lambda x: (float('inf') if not np.isfinite(x[1]) else x[1], x[0])) if np.isfinite(band)), float('nan')),
            'any_gt_inside_em_band': bool(any(np.isfinite(band) and band <= eps for eps, band in neighborhood_rows)),
            'PASS': True,
            'status': 'INFO',
        })
    except Exception as exc:
        rows_em.append(error_row('algorithm23_em_test_cf_sampling_basin', case.graph_id, exc, 'facit_kldm_em_basin'))
safe_display_sorted(dataframe_result('algorithm23_em_test_cf_sampling_basin', rows_em), ['graph'])
safe_display_sorted(dataframe_result('algorithm23_em_test_cf_sampling_basin_epsilon', rows_em_eps), ['graph', 'epsilon'])


## Test Group A — Oracle-Template Local Tests

Decision goal: if the correct Wyckoff chart is known, determine whether CF-MCMC improves q locally without leaving the KLDM geometry basin, and whether that improvement survives the CPS state update.


In [10]:
rows_a1 = []
rows_a2 = []
rows_a2_summary = []
rows_a3 = []
rows_a3_summary = []
rows_a5 = []
rows_a5_summary = []
for case in GRAPH_CASES[:1]:
    payload = build_oracle_payload(case)
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    for step in ALGO23_ORACLE_STEPS:
        state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(step))
        f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
        clean_ev = evaluate_frac(case, f0_hat)
        geom = algorithm23_oracle_geom_fit(case, state, payload, f0_hat)
        geom_ev = evaluate_frac(case, geom.frac_coords_model_order)
        geom_cf = cf_energy_for_algo23(geom.candidate, geom.q_star, state.l, case=case, label=f'g{case.graph_id}_a1_{step}') if CF_READY else float('nan')
        rows_a1.append({
            'test': 'algorithm23_a1_oracle_qfit_baseline',
            'graph': case.graph_id,
            'step': int(step),
            'clean_frac_rmse_baseline': safe_metric_float(clean_ev.get('frac_rmse')),
            'clean_rmse_baseline': safe_metric_float(clean_ev.get('rmse')),
            'geometry_cost': float(geom.witness),
            'cf_nll_q_geom': safe_metric_float(geom_cf),
            'frac_rmse': safe_metric_float(geom_ev.get('frac_rmse')),
            'rmse': safe_metric_float(geom_ev.get('rmse')),
            'geom_beats_clean_frac_rmse': strict_improves(geom_ev.get('frac_rmse'), clean_ev.get('frac_rmse')),
            'geom_beats_clean_rmse': strict_improves(geom_ev.get('rmse'), clean_ev.get('rmse')),
            'match': safe_metric_bool(geom_ev.get('match')),
            'valid': safe_metric_bool(geom_ev.get('valid')),
            'q_distance_to_GT': float(torch.sqrt(cf_backend.torus_mse(geom.q_star.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item()) if geom.q_star.numel() == q_gt.numel() else float('nan'),
            'PASS': True,
            'status': 'INFO',
        })
        if not CF_READY:
            continue
        for spec in [
            {'label': 'greedy_s8_tr10', 'mcmc_steps': 8, 'proposal_sigmas': (0.005, 0.01, 0.02), 'trust_region_relative': 0.10, 'acceptance_mode': 'greedy', 'temperature': 1.0},
            {'label': 'greedy_s16_tr10', 'mcmc_steps': 16, 'proposal_sigmas': (0.005, 0.01, 0.02), 'trust_region_relative': 0.10, 'acceptance_mode': 'greedy', 'temperature': 1.0},
        ]:
            cfg23 = make_algo23_cfg(
                mcmc_steps=int(spec['mcmc_steps']),
                proposal_sigmas=tuple(spec['proposal_sigmas']),
                trust_region_relative=float(spec['trust_region_relative']),
                acceptance_mode=str(spec['acceptance_mode']),
                temperature=float(spec['temperature']),
            )
            proposals_result = run_algo23_generate_proposals(case, state, f0_hat, geom.candidate, cfg23, label=f"g{case.graph_id}_a2_{step}_{spec['label']}")
            cf_best_prop = proposal_best_by_cf(proposals_result)
            cf_best_ev = evaluate_frac(case, cf_best_prop.frac_coords_model_order)
            rows_a2.append({
                'test': 'algorithm23_a2_oracle_template_cf_mcmc_local_improvement',
                'graph': case.graph_id,
                'step': int(step),
                'config_label': str(spec['label']),
                'mcmc_steps': int(spec['mcmc_steps']),
                'trust_region_relative': float(spec['trust_region_relative']),
                'acceptance_rate': float(proposals_result.acceptance_rate),
                'num_inside_trust': int(proposals_result.num_inside_trust),
                'num_accepted_steps': int(accepted_count(proposals_result)),
                'chain_moved': bool(chain_moved(proposals_result)),
                'cf_nll_before': safe_metric_float(proposals_result.cf_nll_before),
                'cf_nll_after_best_cf': safe_metric_float(cf_best_prop.cf_nll),
                'geometry_cost_before': float(proposals_result.geometry_cost_before),
                'geometry_cost_best_cf': float(cf_best_prop.geometry_cost),
                'frac_rmse_before': safe_metric_float(geom_ev.get('frac_rmse')),
                'frac_rmse_best_cf': safe_metric_float(cf_best_ev.get('frac_rmse')),
                'rmse_before': safe_metric_float(geom_ev.get('rmse')),
                'rmse_best_cf': safe_metric_float(cf_best_ev.get('rmse')),
                'cf_nll_strictly_improves': strict_improves(cf_best_prop.cf_nll, proposals_result.cf_nll_before),
                'improves_frac_rmse_strict': strict_improves(cf_best_ev.get('frac_rmse'), geom_ev.get('frac_rmse')),
                'improves_rmse_strict': strict_improves(cf_best_ev.get('rmse'), geom_ev.get('rmse')),
                'PASS': True,
                'status': 'INFO',
            })
            for alpha in ALGO23_ALPHA_VALUES:
                selection = run_algo23_survival_selection(state, f0_hat, proposals_result, cfg23, alpha=float(alpha))
                baseline_eval = selection.baseline
                best_survival = selection.best
                cf_eval = survival_eval_for_proposal(selection, cf_best_prop)
                rows_a5.extend([
                    {
                        'test': 'algorithm23_a5_oracle_template_cps_survival',
                        'graph': case.graph_id,
                        'selector': 'q_geom_baseline',
                        'config_label': str(spec['label']),
                        'step': int(step),
                        'alpha': float(alpha),
                        'proposal_source': str(baseline_eval.proposal.source),
                        'proposal_step': int(baseline_eval.proposal.step),
                        'proposal_cf_nll': safe_metric_float(baseline_eval.proposal.cf_nll),
                        'proposal_geometry_cost': float(baseline_eval.proposal.geometry_cost),
                        'post_geometry_cost': float(baseline_eval.post_geometry_cost),
                        'post_frac_rmse': safe_metric_float(evaluate_frac(case, baseline_eval.f0_after).get('frac_rmse')),
                        'post_rmse': safe_metric_float(evaluate_frac(case, baseline_eval.f0_after).get('rmse')),
                        'match': safe_metric_bool(evaluate_frac(case, baseline_eval.f0_after).get('match')),
                        'valid': safe_metric_bool(evaluate_frac(case, baseline_eval.f0_after).get('valid')),
                        'PASS': True,
                        'status': 'INFO',
                    },
                    {
                        'test': 'algorithm23_a5_oracle_template_cps_survival',
                        'graph': case.graph_id,
                        'selector': 'q_best_cf_nll',
                        'config_label': str(spec['label']),
                        'step': int(step),
                        'alpha': float(alpha),
                        'proposal_source': str(cf_best_prop.source),
                        'proposal_step': int(cf_best_prop.step),
                        'proposal_cf_nll': safe_metric_float(cf_best_prop.cf_nll),
                        'proposal_geometry_cost': float(cf_best_prop.geometry_cost),
                        'post_geometry_cost': float(cf_eval.post_geometry_cost) if cf_eval is not None else float('nan'),
                        'post_frac_rmse': safe_metric_float(evaluate_frac(case, cf_eval.f0_after).get('frac_rmse')) if cf_eval is not None else float('nan'),
                        'post_rmse': safe_metric_float(evaluate_frac(case, cf_eval.f0_after).get('rmse')) if cf_eval is not None else float('nan'),
                        'match': safe_metric_bool(evaluate_frac(case, cf_eval.f0_after).get('match')) if cf_eval is not None else False,
                        'valid': safe_metric_bool(evaluate_frac(case, cf_eval.f0_after).get('valid')) if cf_eval is not None else False,
                        'PASS': bool(cf_eval is not None),
                        'status': 'INFO' if cf_eval is not None else 'MISSING_CF_EVAL',
                    },
                    {
                        'test': 'algorithm23_a5_oracle_template_cps_survival',
                        'graph': case.graph_id,
                        'selector': 'q_selected_by_post_update',
                        'config_label': str(spec['label']),
                        'step': int(step),
                        'alpha': float(alpha),
                        'proposal_source': str(best_survival.proposal.source),
                        'proposal_step': int(best_survival.proposal.step),
                        'proposal_cf_nll': safe_metric_float(best_survival.proposal.cf_nll),
                        'proposal_geometry_cost': float(best_survival.proposal.geometry_cost),
                        'post_geometry_cost': float(best_survival.post_geometry_cost),
                        'post_frac_rmse': safe_metric_float(evaluate_frac(case, best_survival.f0_after).get('frac_rmse')),
                        'post_rmse': safe_metric_float(evaluate_frac(case, best_survival.f0_after).get('rmse')),
                        'match': safe_metric_bool(evaluate_frac(case, best_survival.f0_after).get('match')),
                        'valid': safe_metric_bool(evaluate_frac(case, best_survival.f0_after).get('valid')),
                        'PASS': True,
                        'status': 'INFO',
                    },
                ])

    stress_step = int(ALGO23_ORACLE_STRESS_STEP)
    state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(stress_step))
    f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
    geom = algorithm23_oracle_geom_fit(case, state, payload, f0_hat)
    geom_ev = evaluate_frac(case, geom.frac_coords_model_order)
    for spec in ALGO23_ORACLE_STRESS_CONFIGS:
        cfg23 = make_algo23_cfg(
            mcmc_steps=int(spec['mcmc_steps']),
            proposal_sigmas=tuple(spec['proposal_sigmas']),
            trust_region_relative=float(spec['trust_region_relative']),
            acceptance_mode=str(spec['acceptance_mode']),
            temperature=float(spec['temperature']),
        )
        proposals_result = run_algo23_generate_proposals(case, state, f0_hat, geom.candidate, cfg23, label=f"g{case.graph_id}_a3_{spec['label']}")
        cf_best_prop = proposal_best_by_cf(proposals_result)
        cf_best_ev = evaluate_frac(case, cf_best_prop.frac_coords_model_order)
        rows_a3.append({
            'test': 'algorithm23_a3_oracle_template_cf_mcmc_stress',
            'graph': case.graph_id,
            'step': int(stress_step),
            'config_label': str(spec['label']),
            'mcmc_steps': int(spec['mcmc_steps']),
            'proposal_sigmas': str(tuple(spec['proposal_sigmas'])),
            'trust_region_relative': float(spec['trust_region_relative']),
            'acceptance_mode': str(spec['acceptance_mode']),
            'temperature': float(spec['temperature']),
            'acceptance_rate': float(proposals_result.acceptance_rate),
            'num_inside_trust': int(proposals_result.num_inside_trust),
            'num_accepted_steps': int(accepted_count(proposals_result)),
            'chain_moved': bool(chain_moved(proposals_result)),
            'cf_nll_before': safe_metric_float(proposals_result.cf_nll_before),
            'cf_nll_best_cf': safe_metric_float(cf_best_prop.cf_nll),
            'geometry_cost_before': float(proposals_result.geometry_cost_before),
            'geometry_cost_best_cf': float(cf_best_prop.geometry_cost),
            'frac_rmse_before': safe_metric_float(geom_ev.get('frac_rmse')),
            'frac_rmse_best_cf': safe_metric_float(cf_best_ev.get('frac_rmse')),
            'rmse_before': safe_metric_float(geom_ev.get('rmse')),
            'rmse_best_cf': safe_metric_float(cf_best_ev.get('rmse')),
            'cf_nll_strictly_improves': strict_improves(cf_best_prop.cf_nll, proposals_result.cf_nll_before),
            'improves_frac_rmse_strict': strict_improves(cf_best_ev.get('frac_rmse'), geom_ev.get('frac_rmse')),
            'improves_rmse_strict': strict_improves(cf_best_ev.get('rmse'), geom_ev.get('rmse')),
            'PASS': True,
            'status': 'INFO',
        })

df_a2_tmp = pd.DataFrame(rows_a2)
if len(df_a2_tmp):
    for step, grp in df_a2_tmp.groupby('step'):
        rows_a2_summary.append({
            'test': 'algorithm23_a2_oracle_template_cf_mcmc_summary',
            'graph': int(grp['graph'].iloc[0]),
            'step': int(step),
            'any_chain_moved': bool(grp['chain_moved'].any()),
            'any_cf_nll_improved': bool(grp['cf_nll_strictly_improves'].any()),
            'cf_helped_frac_rmse_strict': bool(grp['improves_frac_rmse_strict'].any()),
            'cf_helped_rmse_strict': bool(grp['improves_rmse_strict'].any()),
            'PASS': bool(grp['chain_moved'].any() and grp['cf_nll_strictly_improves'].any() and (grp['improves_frac_rmse_strict'].any() or grp['improves_rmse_strict'].any())),
            'status': 'INFO',
        })

df_a3_tmp = pd.DataFrame(rows_a3)
if len(df_a3_tmp):
    rows_a3_summary.append({
        'test': 'algorithm23_a3_oracle_template_cf_mcmc_stress_summary',
        'graph': int(df_a3_tmp['graph'].iloc[0]),
        'step': int(df_a3_tmp['step'].iloc[0]),
        'num_configs': int(len(df_a3_tmp)),
        'num_chain_moved': int(df_a3_tmp['chain_moved'].sum()),
        'num_cf_nll_improved': int(df_a3_tmp['cf_nll_strictly_improves'].sum()),
        'num_frac_rmse_improved': int(df_a3_tmp['improves_frac_rmse_strict'].sum()),
        'num_rmse_improved': int(df_a3_tmp['improves_rmse_strict'].sum()),
        'PASS': bool(df_a3_tmp['chain_moved'].any() and df_a3_tmp['cf_nll_strictly_improves'].any()),
        'status': 'INFO',
    })

df_a5_tmp = pd.DataFrame(rows_a5)
if len(df_a5_tmp):
    for (step, alpha, config_label), grp in df_a5_tmp.groupby(['step', 'alpha', 'config_label']):
        geom_row = grp[grp['selector'] == 'q_geom_baseline'].iloc[0]
        cf_row = grp[grp['selector'] == 'q_best_cf_nll'].iloc[0]
        survival_row = grp[grp['selector'] == 'q_selected_by_post_update'].iloc[0]
        rows_a5_summary.append({
            'test': 'algorithm23_a5_oracle_template_cps_survival_summary',
            'graph': int(grp['graph'].iloc[0]),
            'step': int(step),
            'alpha': float(alpha),
            'config_label': str(config_label),
            'geom_post_frac_rmse': safe_metric_float(geom_row['post_frac_rmse']),
            'cf_best_nll_post_frac_rmse': safe_metric_float(cf_row['post_frac_rmse']),
            'survival_post_frac_rmse': safe_metric_float(survival_row['post_frac_rmse']),
            'geom_post_rmse': safe_metric_float(geom_row['post_rmse']),
            'cf_best_nll_post_rmse': safe_metric_float(cf_row['post_rmse']),
            'survival_post_rmse': safe_metric_float(survival_row['post_rmse']),
            'best_by_cf_same_as_best_by_survival': bool(str(cf_row['proposal_source']) == str(survival_row['proposal_source']) and int(cf_row['proposal_step']) == int(survival_row['proposal_step'])),
            'cf_best_vs_geom_frac_rmse': compare_metric(cf_row['post_frac_rmse'], geom_row['post_frac_rmse']),
            'survival_vs_geom_frac_rmse': compare_metric(survival_row['post_frac_rmse'], geom_row['post_frac_rmse']),
            'cf_best_vs_geom_rmse': compare_metric(cf_row['post_rmse'], geom_row['post_rmse']),
            'survival_vs_geom_rmse': compare_metric(survival_row['post_rmse'], geom_row['post_rmse']),
            'survival_beats_geom_frac_rmse_strict': strict_improves(survival_row['post_frac_rmse'], geom_row['post_frac_rmse']),
            'survival_beats_geom_rmse_strict': strict_improves(survival_row['post_rmse'], geom_row['post_rmse']),
            'PASS': bool(strict_improves(survival_row['post_frac_rmse'], geom_row['post_frac_rmse']) or strict_improves(survival_row['post_rmse'], geom_row['post_rmse'])),
            'status': 'INFO',
        })

safe_display_sorted(dataframe_result('algorithm23_a1_oracle_qfit_baseline', rows_a1), ['graph', 'step'])
safe_display_sorted(dataframe_result('algorithm23_a2_oracle_template_cf_mcmc_local_improvement', rows_a2), ['graph', 'step', 'config_label'])
safe_display_sorted(dataframe_result('algorithm23_a2_oracle_template_cf_mcmc_summary', rows_a2_summary), ['graph', 'step'])
safe_display_sorted(dataframe_result('algorithm23_a3_oracle_template_cf_mcmc_stress', rows_a3), ['graph', 'config_label'])
safe_display_sorted(dataframe_result('algorithm23_a3_oracle_template_cf_mcmc_stress_summary', rows_a3_summary), ['graph'])
safe_display_sorted(dataframe_result('algorithm23_a5_oracle_template_cps_survival', rows_a5), ['graph', 'step', 'alpha', 'config_label', 'selector'])
safe_display_sorted(dataframe_result('algorithm23_a5_oracle_template_cps_survival_summary', rows_a5_summary), ['graph', 'step', 'alpha', 'config_label'])


[cf-eval-batch-algo23] worker done g2_a1_600 n=1
[cf-eval-batch-algo23] worker done g2_a2_600_greedy_s8_tr10 n=1
[cf-eval-batch-algo23] worker done g2_a2_600_greedy_s16_tr10 n=1
[cf-eval-batch-algo23] worker done g2_a1_700 n=1
[cf-eval-batch-algo23] worker done g2_a2_700_greedy_s8_tr10 n=1
[cf-eval-batch-algo23] worker done g2_a2_700_greedy_s16_tr10 n=1
[cf-eval-batch-algo23] worker done g2_a1_750 n=1
[cf-eval-batch-algo23] worker done g2_a2_750_greedy_s8_tr10 n=1
[cf-eval-batch-algo23] worker done g2_a2_750_greedy_s16_tr10 n=1
[cf-eval-batch-algo23] worker done g2_a3_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_a3_greedy_small_tr10_s32 n=1
[cf-eval-batch-algo23] worker done g2_a3_greedy_wide_tr20_s32 n=1
[cf-eval-batch-algo23] worker done g2_a3_metro_wide_tr20_s32 n=1


,test,graph,step,clean_frac_rmse_baseline,clean_rmse_baseline,geometry_cost,cf_nll_q_geom,frac_rmse,rmse,geom_beats_clean_frac_rmse,geom_beats_clean_rmse,match,valid,q_distance_to_GT,PASS,status
0,algorithm23_a1_oracle_qfit_baseline,2,600,0.004279,0.018199,0.000376,61.508625,0.006901,0.027663,False,False,True,True,0.328974,True,INFO
1,algorithm23_a1_oracle_qfit_baseline,2,700,0.003649,0.015172,0.000362,58.181782,0.006389,0.025464,False,False,True,True,0.329030,True,INFO
2,algorithm23_a1_oracle_qfit_baseline,2,750,0.002140,0.009221,0.000340,59.151550,0.005608,0.022293,False,False,True,True,0.329217,True,INFO


,test,graph,step,config_label,mcmc_steps,trust_region_relative,acceptance_rate,num_inside_trust,num_accepted_steps,chain_moved,...,geometry_cost_best_cf,frac_rmse_before,frac_rmse_best_cf,rmse_before,rmse_best_cf,cf_nll_strictly_improves,improves_frac_rmse_strict,improves_rmse_strict,PASS,status
0,algorithm23_a2_oracle_template_cf_mcmc_local_i...,2,600,greedy_s16_tr10,16,0.1,0.0,0,0,False,...,0.000038,0.006901,0.006901,0.027663,0.027663,False,False,False,True,INFO
1,algorithm23_a2_oracle_template_cf_mcmc_local_i...,2,600,greedy_s8_tr10,8,0.1,0.0,0,0,False,...,0.000038,0.006901,0.006901,0.027663,0.027663,False,False,False,True,INFO
2,algorithm23_a2_oracle_template_cf_mcmc_local_i...,2,700,greedy_s16_tr10,16,0.1,0.0,0,0,False,...,0.000037,0.006389,0.006389,0.025464,0.025464,False,False,False,True,INFO
3,algorithm23_a2_oracle_template_cf_mcmc_local_i...,2,700,greedy_s8_tr10,8,0.1,0.0,0,0,False,...,0.000037,0.006389,0.006389,0.025464,0.025464,False,False,False,True,INFO
4,algorithm23_a2_oracle_template_cf_mcmc_local_i...,2,750,greedy_s16_tr10,16,0.1,0.0,0,0,False,...,0.000034,0.005608,0.005608,0.022293,0.022293,False,False,False,True,INFO
5,algorithm23_a2_oracle_template_cf_mcmc_local_i...,2,750,greedy_s8_tr10,8,0.1,0.0,0,0,False,...,0.000034,0.005608,0.005608,0.022293,0.022293,False,False,False,True,INFO


,test,graph,step,any_chain_moved,any_cf_nll_improved,cf_helped_frac_rmse_strict,cf_helped_rmse_strict,PASS,status
0,algorithm23_a2_oracle_template_cf_mcmc_summary,2,600,False,False,False,False,False,INFO
1,algorithm23_a2_oracle_template_cf_mcmc_summary,2,700,False,False,False,False,False,INFO
2,algorithm23_a2_oracle_template_cf_mcmc_summary,2,750,False,False,False,False,False,INFO


,test,graph,step,config_label,mcmc_steps,proposal_sigmas,trust_region_relative,acceptance_mode,temperature,acceptance_rate,...,geometry_cost_best_cf,frac_rmse_before,frac_rmse_best_cf,rmse_before,rmse_best_cf,cf_nll_strictly_improves,improves_frac_rmse_strict,improves_rmse_strict,PASS,status
0,algorithm23_a3_oracle_template_cf_mcmc_stress,2,750,greedy_small_tr10_s32,32,"(0.005, 0.01, 0.02)",0.1,greedy,1.0,0.0,...,0.000034,0.005608,0.005608,0.022293,0.022293,False,False,False,True,INFO
1,algorithm23_a3_oracle_template_cf_mcmc_stress,2,750,greedy_small_tr10_s8,8,"(0.005, 0.01, 0.02)",0.1,greedy,1.0,0.0,...,0.000034,0.005608,0.005608,0.022293,0.022293,False,False,False,True,INFO
2,algorithm23_a3_oracle_template_cf_mcmc_stress,2,750,greedy_wide_tr20_s32,32,"(0.01, 0.02, 0.05)",0.2,greedy,1.0,0.0,...,0.000034,0.005608,0.005608,0.022293,0.022293,False,False,False,True,INFO
3,algorithm23_a3_oracle_template_cf_mcmc_stress,2,750,metro_wide_tr20_s32,32,"(0.01, 0.02, 0.05)",0.2,metropolis,1.0,0.0,...,0.000034,0.005608,0.005608,0.022293,0.022293,False,False,False,True,INFO


,test,graph,step,num_configs,num_chain_moved,num_cf_nll_improved,num_frac_rmse_improved,num_rmse_improved,PASS,status
0,algorithm23_a3_oracle_template_cf_mcmc_stress_...,2,750,4,0,0,0,0,False,INFO


,test,graph,selector,config_label,step,alpha,proposal_source,proposal_step,proposal_cf_nll,proposal_geometry_cost,post_geometry_cost,post_frac_rmse,post_rmse,match,valid,PASS,status
0,algorithm23_a5_oracle_template_cps_survival,2,q_best_cf_nll,greedy_s16_tr10,600,0.05,q_geom,-1,61.508625,0.000038,0.000037,0.004265,0.018123,True,True,True,INFO
1,algorithm23_a5_oracle_template_cps_survival,2,q_geom_baseline,greedy_s16_tr10,600,0.05,q_geom,-1,61.508625,0.000038,0.000037,0.004265,0.018123,True,True,True,INFO
2,algorithm23_a5_oracle_template_cps_survival,2,q_selected_by_post_update,greedy_s16_tr10,600,0.05,q_geom,-1,61.508625,0.000038,0.000037,0.004265,0.018123,True,True,True,INFO
3,algorithm23_a5_oracle_template_cps_survival,2,q_best_cf_nll,greedy_s8_tr10,600,0.05,q_geom,-1,61.508625,0.000038,0.000037,0.004265,0.018123,True,True,True,INFO
4,algorithm23_a5_oracle_template_cps_survival,2,q_geom_baseline,greedy_s8_tr10,600,0.05,q_geom,-1,61.508625,0.000038,0.000037,0.004265,0.018123,True,True,True,INFO
5,algorithm23_a5_oracle_template_cps_survival,2,q_selected_by_post_update,greedy_s8_tr10,600,0.05,q_geom,-1,61.508625,0.000038,0.000037,0.004265,0.018123,True,True,True,INFO
6,algorithm23_a5_oracle_template_cps_survival,2,q_best_cf_nll,greedy_s16_tr10,600,0.10,q_geom,-1,61.508625,0.000038,0.000036,0.004288,0.018210,True,True,True,INFO
7,algorithm23_a5_oracle_template_cps_survival,2,q_geom_baseline,greedy_s16_tr10,600,0.10,q_geom,-1,61.508625,0.000038,0.000036,0.004288,0.018210,True,True,True,INFO
8,algorithm23_a5_oracle_template_cps_survival,2,q_selected_by_post_update,greedy_s16_tr10,600,0.10,q_geom,-1,61.508625,0.000038,0.000036,0.004288,0.018210,True,True,True,INFO
9,algorithm23_a5_oracle_template_cps_survival,2,q_best_cf_nll,greedy_s8_tr10,600,0.10,q_geom,-1,61.508625,0.000038,0.000036,0.004288,0.018210,True,True,True,INFO


,test,graph,step,alpha,config_label,geom_post_frac_rmse,cf_best_nll_post_frac_rmse,survival_post_frac_rmse,geom_post_rmse,cf_best_nll_post_rmse,survival_post_rmse,best_by_cf_same_as_best_by_survival,cf_best_vs_geom_frac_rmse,survival_vs_geom_frac_rmse,cf_best_vs_geom_rmse,survival_vs_geom_rmse,survival_beats_geom_frac_rmse_strict,survival_beats_geom_rmse_strict,PASS,status
0,algorithm23_a5_oracle_template_cps_survival_su...,2,600,0.05,greedy_s16_tr10,0.004265,0.004265,0.004265,0.018123,0.018123,0.018123,True,equal,equal,equal,equal,False,False,False,INFO
1,algorithm23_a5_oracle_template_cps_survival_su...,2,600,0.05,greedy_s8_tr10,0.004265,0.004265,0.004265,0.018123,0.018123,0.018123,True,equal,equal,equal,equal,False,False,False,INFO
2,algorithm23_a5_oracle_template_cps_survival_su...,2,600,0.10,greedy_s16_tr10,0.004288,0.004288,0.004288,0.018210,0.018210,0.018210,True,equal,equal,equal,equal,False,False,False,INFO
3,algorithm23_a5_oracle_template_cps_survival_su...,2,600,0.10,greedy_s8_tr10,0.004288,0.004288,0.004288,0.018210,0.018210,0.018210,True,equal,equal,equal,equal,False,False,False,INFO
4,algorithm23_a5_oracle_template_cps_survival_su...,2,700,0.05,greedy_s16_tr10,0.003624,0.003624,0.003624,0.015024,0.015024,0.015024,True,equal,equal,equal,equal,False,False,False,INFO
5,algorithm23_a5_oracle_template_cps_survival_su...,2,700,0.05,greedy_s8_tr10,0.003624,0.003624,0.003624,0.015024,0.015024,0.015024,True,equal,equal,equal,equal,False,False,False,INFO
6,algorithm23_a5_oracle_template_cps_survival_su...,2,700,0.10,greedy_s16_tr10,0.003594,0.003594,0.003594,0.014853,0.014853,0.014853,True,equal,equal,equal,equal,False,False,False,INFO
7,algorithm23_a5_oracle_template_cps_survival_su...,2,700,0.10,greedy_s8_tr10,0.003594,0.003594,0.003594,0.014853,0.014853,0.014853,True,equal,equal,equal,equal,False,False,False,INFO
8,algorithm23_a5_oracle_template_cps_survival_su...,2,750,0.05,greedy_s16_tr10,0.002087,0.002087,0.002087,0.008988,0.008988,0.008988,True,equal,equal,equal,equal,False,False,False,INFO
9,algorithm23_a5_oracle_template_cps_survival_su...,2,750,0.05,greedy_s8_tr10,0.002087,0.002087,0.002087,0.008988,0.008988,0.008988,True,equal,equal,equal,equal,False,False,False,INFO


## Test Group B — Space-Group-Only Local Tests

Decision goal: without oracle template access, determine whether PyXtal + KLDM geometry q-fit + CF-MCMC improves the top geometry templates enough to beat the Algorithm22 geometry-only local correction.


In [11]:
rows_b1 = []
rows_b2 = []
rows_b2_summary = []
rows_b3 = []
rows_b3_summary = []
rows_b4 = []
rows_b4_summary = []
rows_b5 = []
rows_b5_summary = []
for case in GRAPH_CASES[:1]:
    state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(700))
    f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
    clean_ev = evaluate_frac(case, f0_hat)
    payload = build_oracle_payload(case)
    q_gt = torch.as_tensor(algo22_backend._get_wyckoff_dof_chart(payload).q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    candidates, _ = generate_source_candidates(case, state, f0_hat, source='pyxtal_only', cfg=make_algo22_cfg(q_sampling_strategies=('pcs_anchor', 'sobol', 'local'), num_q_per_template=8, top_branches=5))
    template_sigs = [template_signature(c.template) for c in candidates if c.template is not None]
    rows_b1.append({
        'test': 'algorithm23_b1_pyxtal_template_enumeration',
        'graph': case.graph_id,
        'num_templates': int(len(set(template_sigs))),
        'oracle_template_found_eval_only': bool(any(sig == oracle_signature(case) for sig in template_sigs)),
        'template_multiplicities_valid': bool(len(template_sigs) > 0),
        'exact_composition_valid': bool(len(candidates) > 0),
        'PASS': bool(len(candidates) > 0),
        'status': 'INFO',
    })
    ranked = rank_candidates_against_clean_estimate_22(f0_hat=f0_hat, state=state, candidates=candidates, config=ALGO22_CFG)
    for rank_idx, item in enumerate(ranked[:5], start=1):
        ev = evaluate_frac(case, item.frac_coords_model_order)
        rows_b2.append({
            'test': 'algorithm23_b2_geometry_qfit_all_templates',
            'graph': case.graph_id,
            'template_rank': int(rank_idx),
            'geometry_cost': float(item.witness),
            'clean_frac_rmse_baseline': safe_metric_float(clean_ev.get('frac_rmse')),
            'clean_rmse_baseline': safe_metric_float(clean_ev.get('rmse')),
            'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
            'rmse': safe_metric_float(ev.get('rmse')),
            'geom_beats_clean_frac_rmse': strict_improves(ev.get('frac_rmse'), clean_ev.get('frac_rmse')),
            'geom_beats_clean_rmse': strict_improves(ev.get('rmse'), clean_ev.get('rmse')),
            'match': safe_metric_bool(ev.get('match')),
            'valid': safe_metric_bool(ev.get('valid')),
            'template_matches_GT_eval_only': bool(template_signature(item.candidate.template) == oracle_signature(case)) if item.candidate.template is not None else False,
            'q_distance_to_GT_eval_only': float(torch.sqrt(cf_backend.torus_mse(item.q_star.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item()) if item.q_star.numel() == q_gt.numel() else float('nan'),
            'PASS': True,
            'status': 'INFO',
        })

    for top_k in ALGO23_TOP_K_TEMPLATES:
        top_candidates = list(ranked[:top_k])
        for cand_idx, item in enumerate(top_candidates):
            before_ev = evaluate_frac(case, item.frac_coords_model_order)
            for spec in ALGO23_NONORACLE_STRESS_CONFIGS:
                cfg23 = make_algo23_cfg(
                    mcmc_steps=int(spec['mcmc_steps']),
                    proposal_sigmas=tuple(spec['proposal_sigmas']),
                    trust_region_relative=float(spec['trust_region_relative']),
                    acceptance_mode=str(spec['acceptance_mode']),
                    temperature=float(spec['temperature']),
                )
                proposals_result = run_algo23_generate_proposals(case, state, f0_hat, item.candidate, cfg23, label=f"g{case.graph_id}_b3_top{top_k}_{cand_idx}_{spec['label']}")
                cf_best_prop = proposal_best_by_cf(proposals_result)
                cf_best_ev = evaluate_frac(case, cf_best_prop.frac_coords_model_order)
                selection = run_algo23_survival_selection(state, f0_hat, proposals_result, cfg23, alpha=0.10)
                best_survival = selection.best
                survival_ev = evaluate_frac(case, best_survival.f0_after)
                baseline_ev = evaluate_frac(case, selection.baseline.f0_after)
                geom_post_frac = safe_metric_float(baseline_ev.get('frac_rmse'))
                geom_post_rmse = safe_metric_float(baseline_ev.get('rmse'))
                cf_post_frac = safe_metric_float(cf_best_ev.get('frac_rmse'))
                cf_post_rmse = safe_metric_float(cf_best_ev.get('rmse'))
                survival_post_frac = safe_metric_float(survival_ev.get('frac_rmse'))
                survival_post_rmse = safe_metric_float(survival_ev.get('rmse'))
                rows_b3.extend([
                    {
                        'test': 'algorithm23_b3_cf_mcmc_topk_geometry_templates',
                        'graph': case.graph_id,
                        'top_k_templates': int(top_k),
                        'candidate_rank': int(cand_idx + 1),
                        'config_label': str(spec['label']),
                        'selector': 'q_geom_baseline',
                        'chain_moved': bool(chain_moved(proposals_result)),
                        'cf_nll_before': safe_metric_float(proposals_result.cf_nll_before),
                        'cf_nll_selected': safe_metric_float(selection.baseline.proposal.cf_nll),
                        'geometry_cost_before': float(proposals_result.geometry_cost_before),
                        'geometry_cost_selected': float(selection.baseline.proposal.geometry_cost),
                        'frac_rmse_before_eval_only': safe_metric_float(before_ev.get('frac_rmse')),
                        'frac_rmse_selected_eval_only': geom_post_frac,
                        'rmse_before_eval_only': safe_metric_float(before_ev.get('rmse')),
                        'rmse_selected_eval_only': geom_post_rmse,
                        'PASS': True,
                        'status': 'INFO',
                    },
                    {
                        'test': 'algorithm23_b3_cf_mcmc_topk_geometry_templates',
                        'graph': case.graph_id,
                        'top_k_templates': int(top_k),
                        'candidate_rank': int(cand_idx + 1),
                        'config_label': str(spec['label']),
                        'selector': 'q_best_cf_nll',
                        'chain_moved': bool(chain_moved(proposals_result)),
                        'cf_nll_before': safe_metric_float(proposals_result.cf_nll_before),
                        'cf_nll_selected': safe_metric_float(cf_best_prop.cf_nll),
                        'geometry_cost_before': float(proposals_result.geometry_cost_before),
                        'geometry_cost_selected': float(cf_best_prop.geometry_cost),
                        'frac_rmse_before_eval_only': safe_metric_float(before_ev.get('frac_rmse')),
                        'frac_rmse_selected_eval_only': cf_post_frac,
                        'rmse_before_eval_only': safe_metric_float(before_ev.get('rmse')),
                        'rmse_selected_eval_only': cf_post_rmse,
                        'PASS': True,
                        'status': 'INFO',
                    },
                    {
                        'test': 'algorithm23_b3_cf_mcmc_topk_geometry_templates',
                        'graph': case.graph_id,
                        'top_k_templates': int(top_k),
                        'candidate_rank': int(cand_idx + 1),
                        'config_label': str(spec['label']),
                        'selector': 'q_selected_by_post_update',
                        'chain_moved': bool(chain_moved(proposals_result)),
                        'cf_nll_before': safe_metric_float(proposals_result.cf_nll_before),
                        'cf_nll_selected': safe_metric_float(best_survival.proposal.cf_nll),
                        'geometry_cost_before': float(proposals_result.geometry_cost_before),
                        'geometry_cost_selected': float(best_survival.proposal.geometry_cost),
                        'frac_rmse_before_eval_only': safe_metric_float(before_ev.get('frac_rmse')),
                        'frac_rmse_selected_eval_only': survival_post_frac,
                        'rmse_before_eval_only': safe_metric_float(before_ev.get('rmse')),
                        'rmse_selected_eval_only': survival_post_rmse,
                        'PASS': True,
                        'status': 'INFO',
                    },
                ])
                proposal_rows = []
                for proposal_index, eval_item in enumerate(selection.evaluations):
                    proposal_ev = evaluate_frac(case, eval_item.proposal.frac_coords_model_order)
                    post_ev = evaluate_frac(case, eval_item.f0_after)
                    proposal_rows.append({
                        'test': 'algorithm23_b4_survival_score_vs_rmse',
                        'graph': case.graph_id,
                        'top_k_templates': int(top_k),
                        'candidate_rank': int(cand_idx + 1),
                        'config_label': str(spec['label']),
                        'proposal_index': int(proposal_index),
                        'proposal_source': str(eval_item.proposal.source),
                        'num_proposals': int(len(selection.evaluations)),
                        'chain_moved': bool(chain_moved(proposals_result)),
                        'survival_score': safe_metric_float(eval_item.score),
                        'post_geometry_cost': safe_metric_float(eval_item.post_geometry_cost),
                        'shift_distance': safe_metric_float(eval_item.shift_distance),
                        'cf_nll': safe_metric_float(eval_item.proposal.cf_nll),
                        'proposal_frac_rmse_eval_only': safe_metric_float(proposal_ev.get('frac_rmse')),
                        'proposal_rmse_eval_only': safe_metric_float(proposal_ev.get('rmse')),
                        'proposal_match_eval_only': safe_metric_bool(proposal_ev.get('match')),
                        'proposal_valid_eval_only': safe_metric_bool(proposal_ev.get('valid')),
                        'post_frac_rmse_eval_only': safe_metric_float(post_ev.get('frac_rmse')),
                        'post_rmse_eval_only': safe_metric_float(post_ev.get('rmse')),
                        'post_match_eval_only': safe_metric_bool(post_ev.get('match')),
                        'post_valid_eval_only': safe_metric_bool(post_ev.get('valid')),
                        'is_geometry_reference': bool(eval_item.proposal.source == 'q_geom'),
                        'is_cf_best': bool(cf_best_prop is not None and eval_item.proposal is cf_best_prop),
                        'is_survival_best': bool(eval_item is best_survival),
                        'PASS': True,
                        'status': 'INFO',
                    })
                rows_b4.extend(proposal_rows)

    for step in [600, 750]:
        state_local = sample_gt_noisy_state(case, t_value=algo22_step_to_t(step))
        f0_local = clean_fractional_estimate_22(state=state_local, model=runner.model, config=ALGO22_CFG)
        pyxtal_local, _ = generate_source_candidates(case, state_local, f0_local, source='pyxtal_only', cfg=ALGO22_CFG)
        ranked_local = rank_candidates_against_clean_estimate_22(f0_hat=f0_local, state=state_local, candidates=pyxtal_local, config=ALGO22_CFG)
        if ranked_local:
            geom_item = ranked_local[0]
            geom_ev = evaluate_frac(case, geom_item.frac_coords_model_order)
            for alpha in ALGO23_ALPHA_VALUES:
                geom_cfg = make_algo23_cfg(mcmc_steps=8)
                update, _f0_after, after_ev = run_algo23_local_update(case, state_local, f0_local, geom_item.frac_coords_model_order, alpha, geom_cfg)
                rows_b5.append({
                    'test': 'algorithm23_b5_local_cps_update_non_oracle_templates',
                    'graph': case.graph_id,
                    'method': 'pyxtal_geom_q',
                    'config_label': 'geometry_reference',
                    'step': int(step),
                    'alpha': float(alpha),
                    'candidate_geometry_cost': float(geom_item.witness),
                    'candidate_cf_nll': safe_metric_float(geom_item.cf_nll),
                    'candidate_frac_rmse_eval_only': safe_metric_float(geom_ev.get('frac_rmse')),
                    'candidate_rmse_eval_only': safe_metric_float(geom_ev.get('rmse')),
                    'post_frac_rmse_eval_only': safe_metric_float(after_ev.get('frac_rmse')),
                    'post_rmse_eval_only': safe_metric_float(after_ev.get('rmse')),
                    'post_match_eval_only': safe_metric_bool(after_ev.get('match')),
                    'post_valid_eval_only': safe_metric_bool(after_ev.get('valid')),
                    'runtime': float('nan'),
                    'PASS': True,
                    'status': 'INFO',
                })
            for spec in ALGO23_NONORACLE_STRESS_CONFIGS:
                cfg23 = make_algo23_cfg(
                    mcmc_steps=int(spec['mcmc_steps']),
                    proposal_sigmas=tuple(spec['proposal_sigmas']),
                    trust_region_relative=float(spec['trust_region_relative']),
                    acceptance_mode=str(spec['acceptance_mode']),
                    temperature=float(spec['temperature']),
                )
                proposals_result = run_algo23_generate_proposals(case, state_local, f0_local, geom_item.candidate, cfg23, label=f"g{case.graph_id}_b5_{step}_{spec['label']}")
                cf_best_prop = proposal_best_by_cf(proposals_result)
                for alpha in ALGO23_ALPHA_VALUES:
                    selection = run_algo23_survival_selection(state_local, f0_local, proposals_result, cfg23, alpha=float(alpha))
                    best_survival = selection.best
                    cf_eval = survival_eval_for_proposal(selection, cf_best_prop)
                    for method, prop_like, eval_like in [
                        ('q_best_cf_nll', cf_best_prop, cf_eval),
                        ('q_selected_by_post_update', best_survival.proposal, best_survival),
                    ]:
                        after_ev = evaluate_frac(case, eval_like.f0_after) if eval_like is not None else {}
                        rows_b5.append({
                            'test': 'algorithm23_b5_local_cps_update_non_oracle_templates',
                            'graph': case.graph_id,
                            'method': method,
                            'config_label': str(spec['label']),
                            'step': int(step),
                            'alpha': float(alpha),
                            'candidate_geometry_cost': float(prop_like.geometry_cost),
                            'candidate_cf_nll': safe_metric_float(prop_like.cf_nll),
                            'candidate_frac_rmse_eval_only': safe_metric_float(evaluate_frac(case, prop_like.frac_coords_model_order).get('frac_rmse')),
                            'candidate_rmse_eval_only': safe_metric_float(evaluate_frac(case, prop_like.frac_coords_model_order).get('rmse')),
                            'post_frac_rmse_eval_only': safe_metric_float(after_ev.get('frac_rmse')),
                            'post_rmse_eval_only': safe_metric_float(after_ev.get('rmse')),
                            'post_match_eval_only': safe_metric_bool(after_ev.get('match')),
                            'post_valid_eval_only': safe_metric_bool(after_ev.get('valid')),
                            'runtime': float('nan'),
                            'PASS': True,
                            'status': 'INFO' if eval_like is not None else 'MISSING_CF_EVAL',
                        })

df_b2_tmp = pd.DataFrame(rows_b2)
if len(df_b2_tmp):
    for graph, grp in df_b2_tmp.groupby('graph'):
        rows_b2_summary.append({
            'test': 'algorithm23_b2_geometry_qfit_summary',
            'graph': int(graph),
            'top1_frac_rmse': finite_min(grp[grp['template_rank'] == 1]['frac_rmse']) if len(grp[grp['template_rank'] == 1]) else float('nan'),
            'top3_min_frac_rmse': finite_min(grp[grp['template_rank'] <= 3]['frac_rmse']),
            'top1_rmse': finite_min(grp[grp['template_rank'] == 1]['rmse']) if len(grp[grp['template_rank'] == 1]) else float('nan'),
            'top3_min_rmse': finite_min(grp[grp['template_rank'] <= 3]['rmse']),
            'best_geom_beats_clean_frac_rmse': bool(grp['geom_beats_clean_frac_rmse'].any()),
            'best_geom_beats_clean_rmse': bool(grp['geom_beats_clean_rmse'].any()),
            'oracle_template_in_top3_eval_only': bool(grp[grp['template_rank'] <= 3]['template_matches_GT_eval_only'].any()),
            'PASS': bool(len(grp) > 0),
            'status': 'INFO',
        })

df_b3_tmp = pd.DataFrame(rows_b3)
if len(df_b3_tmp):
    for (top_k, config_label), grp in df_b3_tmp.groupby(['top_k_templates', 'config_label']):
        geom_grp = grp[grp['selector'] == 'q_geom_baseline']
        cf_grp = grp[grp['selector'] == 'q_best_cf_nll']
        survival_grp = grp[grp['selector'] == 'q_selected_by_post_update']
        rows_b3_summary.append({
            'test': 'algorithm23_b3_cf_mcmc_topk_summary',
            'graph': int(grp['graph'].iloc[0]),
            'top_k_templates': int(top_k),
            'config_label': str(config_label),
            'chain_moved_rate': bool_rate(grp['chain_moved']),
            'cf_best_vs_geom_frac_rmse': compare_metric(finite_min(cf_grp['frac_rmse_selected_eval_only']), finite_min(geom_grp['frac_rmse_selected_eval_only'])),
            'survival_vs_geom_frac_rmse': compare_metric(finite_min(survival_grp['frac_rmse_selected_eval_only']), finite_min(geom_grp['frac_rmse_selected_eval_only'])),
            'cf_best_vs_geom_rmse': compare_metric(finite_min(cf_grp['rmse_selected_eval_only']), finite_min(geom_grp['rmse_selected_eval_only'])),
            'survival_vs_geom_rmse': compare_metric(finite_min(survival_grp['rmse_selected_eval_only']), finite_min(geom_grp['rmse_selected_eval_only'])),
            'survival_beats_geom_strict': bool(strict_improves(finite_min(survival_grp['frac_rmse_selected_eval_only']), finite_min(geom_grp['frac_rmse_selected_eval_only'])) or strict_improves(finite_min(survival_grp['rmse_selected_eval_only']), finite_min(geom_grp['rmse_selected_eval_only']))),
            'PASS': bool(strict_improves(finite_min(survival_grp['frac_rmse_selected_eval_only']), finite_min(geom_grp['frac_rmse_selected_eval_only'])) or strict_improves(finite_min(survival_grp['rmse_selected_eval_only']), finite_min(geom_grp['rmse_selected_eval_only']))),
            'status': 'INFO',
        })

df_b4_tmp = pd.DataFrame(rows_b4)
if len(df_b4_tmp):
    for (graph, top_k, config_label, candidate_rank), grp in df_b4_tmp.groupby(['graph', 'top_k_templates', 'config_label', 'candidate_rank']):
        grp_sorted_survival = grp.sort_values(['survival_score', 'proposal_frac_rmse_eval_only'], kind='mergesort').reset_index(drop=True)
        grp_sorted_cf = grp.copy()
        grp_sorted_cf['_cf_sort'] = grp_sorted_cf['cf_nll'].where(np.isfinite(grp_sorted_cf['cf_nll']), np.inf)
        grp_sorted_cf = grp_sorted_cf.sort_values(['_cf_sort', 'proposal_frac_rmse_eval_only'], kind='mergesort').reset_index(drop=True)
        geom_rows = grp[grp['is_geometry_reference']]
        survival_top1 = grp_sorted_survival.iloc[0]
        cf_top1 = grp_sorted_cf.iloc[0]
        geom_top1 = geom_rows.iloc[0] if len(geom_rows) else grp_sorted_survival.iloc[0]
        finite_mask_frac = np.isfinite(grp['survival_score']) & np.isfinite(grp['proposal_frac_rmse_eval_only'])
        finite_mask_rmse = np.isfinite(grp['survival_score']) & np.isfinite(grp['proposal_rmse_eval_only'])
        spearman_frac = float(pd.Series(grp.loc[finite_mask_frac, 'survival_score']).rank().corr(pd.Series(grp.loc[finite_mask_frac, 'proposal_frac_rmse_eval_only']).rank(), method='pearson')) if int(finite_mask_frac.sum()) > 1 else float('nan')
        spearman_rmse = float(pd.Series(grp.loc[finite_mask_rmse, 'survival_score']).rank().corr(pd.Series(grp.loc[finite_mask_rmse, 'proposal_rmse_eval_only']).rank(), method='pearson')) if int(finite_mask_rmse.sum()) > 1 else float('nan')
        top3_survival_frac = finite_min(grp_sorted_survival.head(3)['proposal_frac_rmse_eval_only'])
        top3_survival_rmse = finite_min(grp_sorted_survival.head(3)['proposal_rmse_eval_only'])
        rows_b4_summary.append({
            'test': 'algorithm23_b4_survival_score_vs_rmse_summary',
            'graph': int(graph),
            'top_k_templates': int(top_k),
            'config_label': str(config_label),
            'candidate_rank': int(candidate_rank),
            'num_proposals': int(len(grp)),
            'spearman_survival_vs_proposal_frac_rmse': spearman_frac,
            'spearman_survival_vs_proposal_rmse': spearman_rmse,
            'top1_survival_proposal_frac_rmse': safe_metric_float(survival_top1['proposal_frac_rmse_eval_only']),
            'top3_survival_min_proposal_frac_rmse': top3_survival_frac,
            'top1_cf_proposal_frac_rmse': safe_metric_float(cf_top1['proposal_frac_rmse_eval_only']),
            'top1_geometry_proposal_frac_rmse': safe_metric_float(geom_top1['proposal_frac_rmse_eval_only']),
            'top1_survival_proposal_rmse': safe_metric_float(survival_top1['proposal_rmse_eval_only']),
            'top3_survival_min_proposal_rmse': top3_survival_rmse,
            'top1_cf_proposal_rmse': safe_metric_float(cf_top1['proposal_rmse_eval_only']),
            'top1_geometry_proposal_rmse': safe_metric_float(geom_top1['proposal_rmse_eval_only']),
            'top1_survival_beats_geometry_strict': bool(strict_improves(survival_top1['proposal_frac_rmse_eval_only'], geom_top1['proposal_frac_rmse_eval_only']) or strict_improves(survival_top1['proposal_rmse_eval_only'], geom_top1['proposal_rmse_eval_only'])),
            'top1_survival_beats_cf_strict': bool(strict_improves(survival_top1['proposal_frac_rmse_eval_only'], cf_top1['proposal_frac_rmse_eval_only']) or strict_improves(survival_top1['proposal_rmse_eval_only'], cf_top1['proposal_rmse_eval_only'])),
            'top3_survival_beats_geometry_strict': bool(strict_improves(top3_survival_frac, geom_top1['proposal_frac_rmse_eval_only']) or strict_improves(top3_survival_rmse, geom_top1['proposal_rmse_eval_only'])),
            'top3_survival_beats_cf_strict': bool(strict_improves(top3_survival_frac, cf_top1['proposal_frac_rmse_eval_only']) or strict_improves(top3_survival_rmse, cf_top1['proposal_rmse_eval_only'])),
            'survival_correlation_useful': bool((np.isfinite(spearman_frac) and spearman_frac > 0.25) or (np.isfinite(spearman_rmse) and spearman_rmse > 0.25)),
            'PASS': bool((strict_improves(survival_top1['proposal_frac_rmse_eval_only'], geom_top1['proposal_frac_rmse_eval_only']) or strict_improves(survival_top1['proposal_rmse_eval_only'], geom_top1['proposal_rmse_eval_only']) or strict_improves(top3_survival_frac, geom_top1['proposal_frac_rmse_eval_only']) or strict_improves(top3_survival_rmse, geom_top1['proposal_rmse_eval_only'])) and ((np.isfinite(spearman_frac) and spearman_frac > 0.0) or (np.isfinite(spearman_rmse) and spearman_rmse > 0.0))),
            'status': 'INFO',
        })

df_b5_tmp = pd.DataFrame(rows_b5)
if len(df_b5_tmp):
    geom_ref = df_b5_tmp[df_b5_tmp['method'] == 'pyxtal_geom_q']
    cf_rows = df_b5_tmp[df_b5_tmp['method'] != 'pyxtal_geom_q']
    for (graph, step, alpha), geom_grp in geom_ref.groupby(['graph', 'step', 'alpha']):
        geom_row = geom_grp.iloc[0]
        matching_cf = cf_rows[(cf_rows['graph'] == graph) & (cf_rows['step'] == step) & (cf_rows['alpha'] == alpha)]
        for _, cf_row in matching_cf.iterrows():
            rows_b5_summary.append({
                'test': 'algorithm23_b5_local_cps_update_summary',
                'graph': int(graph),
                'step': int(step),
                'alpha': float(alpha),
                'config_label': str(cf_row['config_label']),
                'selector': str(cf_row['method']),
                'geom_post_frac_rmse': safe_metric_float(geom_row['post_frac_rmse_eval_only']),
                'cf_post_frac_rmse': safe_metric_float(cf_row['post_frac_rmse_eval_only']),
                'geom_post_rmse': safe_metric_float(geom_row['post_rmse_eval_only']),
                'cf_post_rmse': safe_metric_float(cf_row['post_rmse_eval_only']),
                'cf_vs_geom_frac_rmse': compare_metric(cf_row['post_frac_rmse_eval_only'], geom_row['post_frac_rmse_eval_only']),
                'cf_vs_geom_rmse': compare_metric(cf_row['post_rmse_eval_only'], geom_row['post_rmse_eval_only']),
                'cf_beats_geom_frac_rmse_strict': strict_improves(cf_row['post_frac_rmse_eval_only'], geom_row['post_frac_rmse_eval_only']),
                'cf_beats_geom_rmse_strict': strict_improves(cf_row['post_rmse_eval_only'], geom_row['post_rmse_eval_only']),
                'PASS': bool(strict_improves(cf_row['post_frac_rmse_eval_only'], geom_row['post_frac_rmse_eval_only']) or strict_improves(cf_row['post_rmse_eval_only'], geom_row['post_rmse_eval_only'])),
                'status': 'INFO',
            })

safe_display_sorted(dataframe_result('algorithm23_b1_pyxtal_template_enumeration', rows_b1), ['graph'])
safe_display_sorted(dataframe_result('algorithm23_b2_geometry_qfit_all_templates', rows_b2), ['graph', 'template_rank'])
safe_display_sorted(dataframe_result('algorithm23_b2_geometry_qfit_summary', rows_b2_summary), ['graph'])
safe_display_sorted(dataframe_result('algorithm23_b3_cf_mcmc_topk_geometry_templates', rows_b3), ['graph', 'top_k_templates', 'config_label', 'selector'])
safe_display_sorted(dataframe_result('algorithm23_b3_cf_mcmc_topk_summary', rows_b3_summary), ['graph', 'top_k_templates', 'config_label'])
safe_display_sorted(dataframe_result('algorithm23_b4_survival_score_vs_rmse', rows_b4), ['graph', 'top_k_templates', 'config_label'])
safe_display_sorted(dataframe_result('algorithm23_b4_survival_score_vs_rmse_summary', rows_b4_summary), ['graph', 'top_k_templates', 'config_label'])
safe_display_sorted(dataframe_result('algorithm23_b5_local_cps_update_non_oracle_templates', rows_b5), ['graph', 'step', 'alpha', 'config_label', 'method'])
safe_display_sorted(dataframe_result('algorithm23_b5_local_cps_update_summary', rows_b5_summary), ['graph', 'step', 'alpha', 'config_label', 'selector'])


[cf-eval-batch-algo23] worker done g2_b3_top1_0_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b3_top1_0_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b3_top1_0_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b3_top1_0_greedy_wide_tr20_s16 n=1
[cf-eval-batch-algo23] worker done g2_b3_top3_0_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b3_top3_0_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b3_top3_0_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b3_top3_0_greedy_wide_tr20_s16 n=1
[cf-eval-batch-algo23] worker done g2_b5_600_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b5_600_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b5_600_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b5_600_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b5_600_greedy_small_tr10_s8 n=1
[cf-eval-batch-algo23] worker done g2_b5_600_greedy_wide_tr20_s16 

,test,graph,num_templates,oracle_template_found_eval_only,template_multiplicities_valid,exact_composition_valid,PASS,status
0,algorithm23_b1_pyxtal_template_enumeration,2,1,True,True,True,True,INFO


,test,graph,template_rank,geometry_cost,clean_frac_rmse_baseline,clean_rmse_baseline,frac_rmse,rmse,geom_beats_clean_frac_rmse,geom_beats_clean_rmse,match,valid,template_matches_GT_eval_only,q_distance_to_GT_eval_only,PASS,status
0,algorithm23_b2_geometry_qfit_all_templates,2,1,0.001239,0.003649,0.015172,0.011455,0.045046,False,False,True,True,True,0.313512,True,INFO


,test,graph,top1_frac_rmse,top3_min_frac_rmse,top1_rmse,top3_min_rmse,best_geom_beats_clean_frac_rmse,best_geom_beats_clean_rmse,oracle_template_in_top3_eval_only,PASS,status
0,algorithm23_b2_geometry_qfit_summary,2,0.011455,0.011455,0.045046,0.045046,False,False,True,True,INFO


,test,graph,top_k_templates,candidate_rank,config_label,selector,chain_moved,cf_nll_before,cf_nll_selected,geometry_cost_before,geometry_cost_selected,frac_rmse_before_eval_only,frac_rmse_selected_eval_only,rmse_before_eval_only,rmse_selected_eval_only,PASS,status
0,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,1,1,greedy_small_tr10_s8,q_best_cf_nll,True,26.197309,25.556894,0.000126,0.000134,0.011455,0.012752,0.045046,0.052091,True,INFO
1,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,1,1,greedy_small_tr10_s8,q_geom_baseline,True,26.197309,26.197309,0.000126,0.000126,0.011455,0.003752,0.045046,0.015507,True,INFO
2,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,1,1,greedy_small_tr10_s8,q_selected_by_post_update,True,26.197309,26.197309,0.000126,0.000126,0.011455,0.003752,0.045046,0.015507,True,INFO
3,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,1,1,greedy_wide_tr20_s16,q_best_cf_nll,False,26.197309,26.197309,0.000126,0.000126,0.011455,0.011455,0.045046,0.045046,True,INFO
4,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,1,1,greedy_wide_tr20_s16,q_geom_baseline,False,26.197309,26.197309,0.000126,0.000126,0.011455,0.003752,0.045046,0.015507,True,INFO
5,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,1,1,greedy_wide_tr20_s16,q_selected_by_post_update,False,26.197309,26.197309,0.000126,0.000126,0.011455,0.003752,0.045046,0.015507,True,INFO
6,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,3,1,greedy_small_tr10_s8,q_best_cf_nll,True,26.197309,25.556894,0.000126,0.000134,0.011455,0.012752,0.045046,0.052091,True,INFO
7,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,3,1,greedy_small_tr10_s8,q_geom_baseline,True,26.197309,26.197309,0.000126,0.000126,0.011455,0.003752,0.045046,0.015507,True,INFO
8,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,3,1,greedy_small_tr10_s8,q_selected_by_post_update,True,26.197309,26.197309,0.000126,0.000126,0.011455,0.003752,0.045046,0.015507,True,INFO
9,algorithm23_b3_cf_mcmc_topk_geometry_templates,2,3,1,greedy_wide_tr20_s16,q_best_cf_nll,False,26.197309,26.197309,0.000126,0.000126,0.011455,0.011455,0.045046,0.045046,True,INFO


,test,graph,top_k_templates,config_label,chain_moved_rate,cf_best_vs_geom_frac_rmse,survival_vs_geom_frac_rmse,cf_best_vs_geom_rmse,survival_vs_geom_rmse,survival_beats_geom_strict,PASS,status
0,algorithm23_b3_cf_mcmc_topk_summary,2,1,greedy_small_tr10_s8,1.0,worse,equal,worse,equal,False,False,INFO
1,algorithm23_b3_cf_mcmc_topk_summary,2,1,greedy_wide_tr20_s16,0.0,worse,equal,worse,equal,False,False,INFO
2,algorithm23_b3_cf_mcmc_topk_summary,2,3,greedy_small_tr10_s8,1.0,worse,equal,worse,equal,False,False,INFO
3,algorithm23_b3_cf_mcmc_topk_summary,2,3,greedy_wide_tr20_s16,0.0,worse,equal,worse,equal,False,False,INFO


,test,graph,top_k_templates,candidate_rank,config_label,proposal_index,proposal_source,num_proposals,chain_moved,survival_score,...,proposal_valid_eval_only,post_frac_rmse_eval_only,post_rmse_eval_only,post_match_eval_only,post_valid_eval_only,is_geometry_reference,is_cf_best,is_survival_best,PASS,status
0,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,0,q_geom,9,True,0.000120,...,True,0.003752,0.015507,True,True,True,False,True,True,INFO
1,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,1,cf_mcmc_accepted,9,True,0.000129,...,True,0.003766,0.015607,True,True,False,True,False,True,INFO
2,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,2,cf_mcmc_inside_rejected,9,True,0.000133,...,True,0.003764,0.015581,True,True,False,False,False,True,INFO
3,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,3,cf_mcmc_rejected,9,True,0.000146,...,True,0.003765,0.015606,True,True,False,False,False,True,INFO
4,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,4,cf_mcmc_rejected,9,True,0.000150,...,True,0.003800,0.015693,True,True,False,False,False,True,INFO
5,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,5,cf_mcmc_rejected,9,True,0.000153,...,True,0.003716,0.015364,True,True,False,False,False,True,INFO
6,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,6,cf_mcmc_rejected,9,True,0.000180,...,True,0.003741,0.015457,True,True,False,False,False,True,INFO
7,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,7,cf_mcmc_rejected,9,True,0.000186,...,True,0.003802,0.015758,True,True,False,False,False,True,INFO
8,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_small_tr10_s8,8,cf_mcmc_rejected,9,True,0.000197,...,True,0.003805,0.015752,True,True,False,False,False,True,INFO
9,algorithm23_b4_survival_score_vs_rmse,2,1,1,greedy_wide_tr20_s16,0,q_geom,17,False,0.000120,...,True,0.003752,0.015507,True,True,True,True,True,True,INFO


,test,graph,top_k_templates,config_label,candidate_rank,num_proposals,spearman_survival_vs_proposal_frac_rmse,spearman_survival_vs_proposal_rmse,top1_survival_proposal_frac_rmse,top3_survival_min_proposal_frac_rmse,...,top3_survival_min_proposal_rmse,top1_cf_proposal_rmse,top1_geometry_proposal_rmse,top1_survival_beats_geometry_strict,top1_survival_beats_cf_strict,top3_survival_beats_geometry_strict,top3_survival_beats_cf_strict,survival_correlation_useful,PASS,status
0,algorithm23_b4_survival_score_vs_rmse_summary,2,1,greedy_small_tr10_s8,1,9,0.833333,0.716667,0.011455,0.011455,...,0.045046,0.052091,0.045046,False,True,False,True,True,False,INFO
1,algorithm23_b4_survival_score_vs_rmse_summary,2,1,greedy_wide_tr20_s16,1,17,0.990196,0.982843,0.011455,0.011455,...,0.045046,0.045046,0.045046,False,False,False,False,True,False,INFO
2,algorithm23_b4_survival_score_vs_rmse_summary,2,3,greedy_small_tr10_s8,1,9,0.833333,0.716667,0.011455,0.011455,...,0.045046,0.052091,0.045046,False,True,False,True,True,False,INFO
3,algorithm23_b4_survival_score_vs_rmse_summary,2,3,greedy_wide_tr20_s16,1,17,0.990196,0.982843,0.011455,0.011455,...,0.045046,0.045046,0.045046,False,False,False,False,True,False,INFO


,test,graph,method,config_label,step,alpha,candidate_geometry_cost,candidate_cf_nll,candidate_frac_rmse_eval_only,candidate_rmse_eval_only,post_frac_rmse_eval_only,post_rmse_eval_only,post_match_eval_only,post_valid_eval_only,runtime,PASS,status
0,algorithm23_b5_local_cps_update_non_oracle_tem...,2,pyxtal_geom_q,geometry_reference,600,0.05,0.001437,NaN,0.012154,0.055459,0.004278,0.018286,True,True,NaN,True,INFO
1,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_best_cf_nll,greedy_small_tr10_s8,600,0.05,0.000124,57.744278,0.011159,0.049879,0.004302,0.018388,True,True,NaN,True,INFO
2,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_selected_by_post_update,greedy_small_tr10_s8,600,0.05,0.000111,60.748749,0.010760,0.048627,0.004333,0.018521,True,True,NaN,True,INFO
3,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_best_cf_nll,greedy_wide_tr20_s16,600,0.05,0.000126,59.274742,0.011563,0.050426,0.004370,0.018653,True,True,NaN,True,INFO
4,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_selected_by_post_update,greedy_wide_tr20_s16,600,0.05,0.000126,59.274742,0.011563,0.050426,0.004370,0.018653,True,True,NaN,True,INFO
5,algorithm23_b5_local_cps_update_non_oracle_tem...,2,pyxtal_geom_q,geometry_reference,600,0.10,0.001437,NaN,0.012154,0.055459,0.004331,0.018478,True,True,NaN,True,INFO
6,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_best_cf_nll,greedy_small_tr10_s8,600,0.10,0.000124,57.744278,0.011159,0.049879,0.004395,0.018785,True,True,NaN,True,INFO
7,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_selected_by_post_update,greedy_small_tr10_s8,600,0.10,0.000111,60.748749,0.010760,0.048627,0.004416,0.018869,True,True,NaN,True,INFO
8,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_best_cf_nll,greedy_wide_tr20_s16,600,0.10,0.000126,59.274742,0.011563,0.050426,0.004509,0.019213,True,True,NaN,True,INFO
9,algorithm23_b5_local_cps_update_non_oracle_tem...,2,q_selected_by_post_update,greedy_wide_tr20_s16,600,0.10,0.000126,59.274742,0.011563,0.050426,0.004509,0.019213,True,True,NaN,True,INFO


,test,graph,step,alpha,config_label,selector,geom_post_frac_rmse,cf_post_frac_rmse,geom_post_rmse,cf_post_rmse,cf_vs_geom_frac_rmse,cf_vs_geom_rmse,cf_beats_geom_frac_rmse_strict,cf_beats_geom_rmse_strict,PASS,status
0,algorithm23_b5_local_cps_update_summary,2,600,0.05,greedy_small_tr10_s8,q_best_cf_nll,0.004278,0.004302,0.018286,0.018388,worse,worse,False,False,False,INFO
1,algorithm23_b5_local_cps_update_summary,2,600,0.05,greedy_small_tr10_s8,q_selected_by_post_update,0.004278,0.004333,0.018286,0.018521,worse,worse,False,False,False,INFO
2,algorithm23_b5_local_cps_update_summary,2,600,0.05,greedy_wide_tr20_s16,q_best_cf_nll,0.004278,0.004370,0.018286,0.018653,worse,worse,False,False,False,INFO
3,algorithm23_b5_local_cps_update_summary,2,600,0.05,greedy_wide_tr20_s16,q_selected_by_post_update,0.004278,0.004370,0.018286,0.018653,worse,worse,False,False,False,INFO
4,algorithm23_b5_local_cps_update_summary,2,600,0.10,greedy_small_tr10_s8,q_best_cf_nll,0.004331,0.004395,0.018478,0.018785,worse,worse,False,False,False,INFO
5,algorithm23_b5_local_cps_update_summary,2,600,0.10,greedy_small_tr10_s8,q_selected_by_post_update,0.004331,0.004416,0.018478,0.018869,worse,worse,False,False,False,INFO
6,algorithm23_b5_local_cps_update_summary,2,600,0.10,greedy_wide_tr20_s16,q_best_cf_nll,0.004331,0.004509,0.018478,0.019213,worse,worse,False,False,False,INFO
7,algorithm23_b5_local_cps_update_summary,2,600,0.10,greedy_wide_tr20_s16,q_selected_by_post_update,0.004331,0.004509,0.018478,0.019213,worse,worse,False,False,False,INFO
8,algorithm23_b5_local_cps_update_summary,2,750,0.05,greedy_small_tr10_s8,q_best_cf_nll,0.002124,0.002124,0.009231,0.009231,equal,equal,False,False,False,INFO
9,algorithm23_b5_local_cps_update_summary,2,750,0.05,greedy_small_tr10_s8,q_selected_by_post_update,0.002124,0.002124,0.009231,0.009231,equal,equal,False,False,False,INFO


## Test Group C — Lightweight Pipeline Verdict

This final block is intentionally lightweight. It does not claim to be a full sampler benchmark; it answers whether the best local Algorithm23 correction beats the Algorithm22 geometry-only baseline and the KLDM baseline on the tested graph.


In [12]:
rows_c = []
rows_c_verdict = []
rows_c2 = []
for case in GRAPH_CASES[:1]:
    baseline = replay_source(case, source='baseline', cfg=ALGO22_CFG)
    geom = replay_source(case, source='pyxtal_only', cfg=ALGO22_CFG)
    rows_c.append({
        'test': 'algorithm23_c1_repeated_seed_sampler',
        'graph': case.graph_id,
        'method': 'baseline_kldm_pc',
        'evidence_level': 'full_baseline_sampler',
        'mean_frac_rmse': float(baseline['frac_rmse']),
        'mean_rmse': float(baseline['rmse']),
        'best_of_k_frac_rmse': float(baseline['best_step_frac_rmse']),
        'best_of_k_rmse': float(baseline['best_step_rmse_to_GT']),
        'match_rate': float(baseline['match_rate']),
        'valid_rate': float(baseline['valid_rate']),
        'group_witness': float(baseline['group_witness']),
        'accepted_fraction': float(baseline['accepted_fraction']),
        'runtime_multiplier': 1.0,
        'PASS': True,
        'status': 'INFO',
    })
    rows_c.append({
        'test': 'algorithm23_c1_repeated_seed_sampler',
        'graph': case.graph_id,
        'method': 'algorithm22b_pyxtal_geom_q',
        'evidence_level': 'full_algorithm22_sampler',
        'mean_frac_rmse': float(geom['frac_rmse']),
        'mean_rmse': float(geom['rmse']),
        'best_of_k_frac_rmse': float(geom['best_step_frac_rmse']),
        'best_of_k_rmse': float(geom['best_step_rmse_to_GT']),
        'match_rate': float(geom['match_rate']),
        'valid_rate': float(geom['valid_rate']),
        'group_witness': float(geom['group_witness']),
        'accepted_fraction': float(geom['accepted_fraction']),
        'runtime_multiplier': float('nan'),
        'PASS': True,
        'status': 'INFO',
    })
    local_summary = result_tables.get('algorithm23_b5_local_cps_update_summary', pd.DataFrame())
    if len(local_summary):
        local_summary = local_summary[(local_summary['selector'] == 'q_selected_by_post_update') & np.isfinite(local_summary['cf_post_frac_rmse'])]
    if len(local_summary):
        best_proxy = local_summary.sort_values(['cf_post_frac_rmse', 'cf_post_rmse']).iloc[0]
        rows_c.append({
            'test': 'algorithm23_c1_repeated_seed_sampler',
            'graph': case.graph_id,
            'method': 'algorithm23_pyxtal_cf_survival_local_proxy_not_full_sampler',
            'evidence_level': 'local_proxy_only',
            'mean_frac_rmse': safe_metric_float(best_proxy['cf_post_frac_rmse']),
            'mean_rmse': safe_metric_float(best_proxy['cf_post_rmse']),
            'best_of_k_frac_rmse': safe_metric_float(best_proxy['cf_post_frac_rmse']),
            'best_of_k_rmse': safe_metric_float(best_proxy['cf_post_rmse']),
            'match_rate': float('nan'),
            'valid_rate': float('nan'),
            'group_witness': float('nan'),
            'accepted_fraction': float('nan'),
            'runtime_multiplier': float('nan'),
            'proxy_config_label': str(best_proxy['config_label']),
            'proxy_selector': str(best_proxy['selector']),
            'PASS': True,
            'status': 'PROXY_ONLY',
        })
        final_step_rows = local_summary[local_summary['step'] == 750]
        if len(final_step_rows):
            best_final = final_step_rows.sort_values(['cf_post_frac_rmse', 'cf_post_rmse']).iloc[0]
            rows_c2.append({
                'test': 'algorithm23_c2_final_step_only_cf_survival_proxy',
                'graph': case.graph_id,
                'step': 750,
                'config_label': str(best_final['config_label']),
                'selector': str(best_final['selector']),
                'algo22_best_of_k_frac_rmse': float(geom['best_step_frac_rmse']),
                'baseline_best_of_k_frac_rmse': float(baseline['best_step_frac_rmse']),
                'final_step_proxy_frac_rmse': safe_metric_float(best_final['cf_post_frac_rmse']),
                'algo22_best_of_k_rmse': float(geom['best_step_rmse_to_GT']),
                'baseline_best_of_k_rmse': float(baseline['best_step_rmse_to_GT']),
                'final_step_proxy_rmse': safe_metric_float(best_final['cf_post_rmse']),
                'proxy_beats_algo22_strict': bool(strict_improves(best_final['cf_post_frac_rmse'], geom['best_step_frac_rmse']) or strict_improves(best_final['cf_post_rmse'], geom['best_step_rmse_to_GT'])),
                'proxy_beats_baseline_strict': bool(strict_improves(best_final['cf_post_frac_rmse'], baseline['best_step_frac_rmse']) or strict_improves(best_final['cf_post_rmse'], baseline['best_step_rmse_to_GT'])),
                'PASS': True,
                'status': 'PROXY_ONLY',
            })
    df_local = pd.DataFrame([r for r in rows_c if r['graph'] == case.graph_id])
    geom_row = df_local[df_local['method'] == 'algorithm22b_pyxtal_geom_q'].iloc[0]
    base_row = df_local[df_local['method'] == 'baseline_kldm_pc'].iloc[0]
    proxy_rows = df_local[df_local['evidence_level'] == 'local_proxy_only']
    best_proxy_frac = finite_min(proxy_rows['best_of_k_frac_rmse']) if len(proxy_rows) else float('nan')
    best_proxy_rmse = finite_min(proxy_rows['best_of_k_rmse']) if len(proxy_rows) else float('nan')
    rows_c_verdict.append({
        'test': 'algorithm23_c1_pipeline_verdict',
        'graph': case.graph_id,
        'baseline_best_of_k_frac_rmse': safe_metric_float(base_row['best_of_k_frac_rmse']),
        'algo22_geom_best_of_k_frac_rmse': safe_metric_float(geom_row['best_of_k_frac_rmse']),
        'algo23_proxy_best_of_k_frac_rmse': safe_metric_float(best_proxy_frac),
        'baseline_best_of_k_rmse': safe_metric_float(base_row['best_of_k_rmse']),
        'algo22_geom_best_of_k_rmse': safe_metric_float(geom_row['best_of_k_rmse']),
        'algo23_proxy_best_of_k_rmse': safe_metric_float(best_proxy_rmse),
        'algo23_proxy_beats_baseline_frac_rmse_strict': strict_improves(best_proxy_frac, base_row['best_of_k_frac_rmse']),
        'algo23_proxy_beats_algo22_geom_frac_rmse_strict': strict_improves(best_proxy_frac, geom_row['best_of_k_frac_rmse']),
        'algo23_proxy_beats_baseline_rmse_strict': strict_improves(best_proxy_rmse, base_row['best_of_k_rmse']),
        'algo23_proxy_beats_algo22_geom_rmse_strict': strict_improves(best_proxy_rmse, geom_row['best_of_k_rmse']),
        'uses_full_algorithm23_sampler': False,
        'recommendation': ('blocked_cf_runtime' if not CF_READY else ('promising_local_cf_survival_proxy_only' if (strict_improves(best_proxy_frac, geom_row['best_of_k_frac_rmse']) or strict_improves(best_proxy_rmse, geom_row['best_of_k_rmse'])) else 'keep_algorithm22_geometry_default')),
        'PASS': bool(np.isfinite(best_proxy_frac)) if len(proxy_rows) else False,
        'status': 'PROXY_ONLY' if len(proxy_rows) else ('BLOCKED_CF_RUNTIME' if not CF_READY else 'NO_PROXY_DATA'),
    })

safe_display_sorted(dataframe_result('algorithm23_c1_repeated_seed_sampler', rows_c), ['graph', 'method'])
safe_display_sorted(dataframe_result('algorithm23_c2_final_step_only_cf_survival_proxy', rows_c2), ['graph'])
safe_display_sorted(dataframe_result('algorithm23_c1_pipeline_verdict', rows_c_verdict), ['graph'])


,test,graph,method,evidence_level,mean_frac_rmse,mean_rmse,best_of_k_frac_rmse,best_of_k_rmse,match_rate,valid_rate,group_witness,accepted_fraction,runtime_multiplier,PASS,status,proxy_config_label,proxy_selector
0,algorithm23_c1_repeated_seed_sampler,2,algorithm22b_pyxtal_geom_q,full_algorithm22_sampler,0.004291,0.018072,0.002500,0.010243,1.0,1.0,0.001221,0.666667,NaN,True,INFO,NaN,NaN
1,algorithm23_c1_repeated_seed_sampler,2,algorithm23_pyxtal_cf_survival_local_proxy_not...,local_proxy_only,0.002114,0.009183,0.002114,0.009183,NaN,NaN,NaN,NaN,NaN,True,PROXY_ONLY,greedy_small_tr10_s8,q_selected_by_post_update
2,algorithm23_c1_repeated_seed_sampler,2,baseline_kldm_pc,full_baseline_sampler,0.004296,0.018215,0.002140,0.009221,1.0,1.0,0.003006,1.000000,1.0,True,INFO,NaN,NaN


,test,graph,step,config_label,selector,algo22_best_of_k_frac_rmse,baseline_best_of_k_frac_rmse,final_step_proxy_frac_rmse,algo22_best_of_k_rmse,baseline_best_of_k_rmse,final_step_proxy_rmse,proxy_beats_algo22_strict,proxy_beats_baseline_strict,PASS,status
0,algorithm23_c2_final_step_only_cf_survival_proxy,2,750,greedy_small_tr10_s8,q_selected_by_post_update,0.0025,0.00214,0.002114,0.010243,0.009221,0.009183,True,True,True,PROXY_ONLY


,test,graph,baseline_best_of_k_frac_rmse,algo22_geom_best_of_k_frac_rmse,algo23_proxy_best_of_k_frac_rmse,baseline_best_of_k_rmse,algo22_geom_best_of_k_rmse,algo23_proxy_best_of_k_rmse,algo23_proxy_beats_baseline_frac_rmse_strict,algo23_proxy_beats_algo22_geom_frac_rmse_strict,algo23_proxy_beats_baseline_rmse_strict,algo23_proxy_beats_algo22_geom_rmse_strict,uses_full_algorithm23_sampler,recommendation,PASS,status
0,algorithm23_c1_pipeline_verdict,2,0.00214,0.0025,0.002114,0.009221,0.010243,0.009183,True,True,True,True,False,promising_local_cf_survival_proxy_only,True,PROXY_ONLY


## Summary

This final table gives a compact view of which Algorithm23 blocks ran and whether they passed their basic feasibility criteria.


## Acceptance Gates

Algorithm23 should only be considered promising if the following gates pass: finite/non-constant CF NLL, stable representation sanity, strong oracle-template geometry q-fit, CF-MCMC local improvement without geometry drift, PyXtal top-k coverage, CF-inside-trust selection matching or beating geometry-only, local CPS survival improvement, and repeated-seed improvement over Algorithm22B.


In [13]:
acceptance_rows = []


def _first_df(name: str):
    df = result_tables.get(name)
    return df if df is not None and len(df) else pd.DataFrame()


cf_smoke = _first_df('algorithm23_smoke_cf_active')
cf_prop = _first_df('algorithm23_smoke_mcmc_proposal_validity')
cf_nll_pass = bool(len(cf_smoke) and len(cf_prop) and bool(cf_smoke['cf_nll_finite'].all()) and bool(cf_prop['cf_varies_nontrivially'].all()))
acceptance_rows.append({'gate': 'CF NLL sanity', 'required_outcome': 'finite, non-constant NLL over q perturbations', 'observed': 'ok' if cf_nll_pass else ('blocked' if len(cf_smoke) and not bool(cf_smoke['cf_active'].all()) else 'flat_or_nonfinite'), 'PASS': cf_nll_pass})

repr_pass = bool(len(cf_smoke) and bool(cf_smoke['representation_stable'].all()))
acceptance_rows.append({'gate': 'Representation sanity', 'required_outcome': 'same q/structure gives stable NLL under expected ordering', 'observed': 'stable_repeat_nll' if repr_pass else ('blocked' if len(cf_smoke) and not bool(cf_smoke['cf_active'].all()) else 'not_stable'), 'PASS': repr_pass})

oracle_fit = _first_df('algorithm23_a1_oracle_qfit_baseline')
oracle_qfit_pass = bool(len(oracle_fit) and (oracle_fit['frac_rmse'] < 0.02).all() and oracle_fit['valid'].all() and oracle_fit['match'].all())
acceptance_rows.append({'gate': 'Oracle-template q-fit', 'required_outcome': 'geometry q-fit reaches good RMSE', 'observed': f"best_frac_rmse={oracle_fit['frac_rmse'].min():.6g}" if len(oracle_fit) else 'missing', 'PASS': oracle_qfit_pass})

oracle_cf = _first_df('algorithm23_a5_oracle_template_cps_survival_summary')
oracle_cf_pass = bool(len(oracle_cf) and oracle_cf['PASS'].any())
acceptance_rows.append({'gate': 'Oracle-template CF-MCMC', 'required_outcome': 'CF proposals selected by KLDM survival beat q_geom', 'observed': 'survival_beats_geom' if oracle_cf_pass else ('missing' if not len(oracle_cf) else 'no_survival_gain'), 'PASS': oracle_cf_pass})

pyxtal_cov = _first_df('algorithm23_b2_geometry_qfit_summary')
pyxtal_cov_pass = bool(len(pyxtal_cov) and (pyxtal_cov['top3_min_frac_rmse'] < 0.03).any())
acceptance_rows.append({'gate': 'PyXtal coverage', 'required_outcome': 'top-k templates include a good candidate', 'observed': f"top3_min_frac_rmse={pyxtal_cov['top3_min_frac_rmse'].min():.6g}" if len(pyxtal_cov) else 'missing', 'PASS': pyxtal_cov_pass})

sel = _first_df('algorithm23_b3_cf_mcmc_topk_summary')
sel_pass = bool(len(sel) and sel['PASS'].any())
acceptance_rows.append({'gate': 'Selection comparison', 'required_outcome': 'KLDM survival selection beats or matches geometry-only', 'observed': 'survival_beats_geom_nonoracle' if sel_pass else ('missing' if not len(sel) else 'no_survival_gain'), 'PASS': sel_pass})

surv_corr = _first_df('algorithm23_b4_survival_score_vs_rmse_summary')
surv_corr_pass = bool(len(surv_corr) and (((surv_corr['top1_survival_beats_geometry_strict']) | (surv_corr['top1_survival_beats_cf_strict']) | (surv_corr['top3_survival_beats_geometry_strict']) | (surv_corr['top3_survival_beats_cf_strict'])) & ((surv_corr['spearman_survival_vs_proposal_frac_rmse'] > 0.0) | (surv_corr['spearman_survival_vs_proposal_rmse'] > 0.0))).any())
acceptance_rows.append({'gate': 'Survival score utility', 'required_outcome': 'survival score correlates with GT RMSE and top-1 or top-3 by survival beats CF/geometry', 'observed': 'survival_correlates_and_selects_well' if surv_corr_pass else ('missing' if not len(surv_corr) else 'survival_not_predictive'), 'PASS': surv_corr_pass})

local_cps = _first_df('algorithm23_b5_local_cps_update_summary')
local_cps_pass = bool(len(local_cps) and local_cps[(local_cps['selector'] == 'q_selected_by_post_update')]['PASS'].any())
acceptance_rows.append({'gate': 'Local CPS update', 'required_outcome': 'weak late survival-selected CF proposals improve post-RMSE', 'observed': 'survival_selected_beats_geom_locally' if local_cps_pass else ('missing' if not len(local_cps) else 'no_local_gain'), 'PASS': local_cps_pass})

verdict = _first_df('algorithm23_c1_pipeline_verdict')
full_pass = bool(len(verdict) and verdict['uses_full_algorithm23_sampler'].any() and ((verdict['algo23_proxy_beats_algo22_geom_frac_rmse_strict']) | (verdict['algo23_proxy_beats_algo22_geom_rmse_strict'])).any())
full_obs = 'actual_full_sampler_beats_algo22' if full_pass else ('proxy_only_not_full_sampler' if len(verdict) and not verdict['uses_full_algorithm23_sampler'].any() else ('blocked' if len(verdict) and (verdict['status'] == 'BLOCKED_CF_RUNTIME').all() else 'not_beaten'))
acceptance_rows.append({'gate': 'Full repeated-seed sampling', 'required_outcome': 'improves mean or best-of-k over Algorithm22B', 'observed': full_obs, 'PASS': full_pass})

acceptance_df = pd.DataFrame(acceptance_rows)
display(acceptance_df)
claim_ready = bool(acceptance_df['PASS'].all()) if len(acceptance_df) else False
display(pd.DataFrame([{'final_gate': 'Algorithm23 claim readiness', 'decision': 'ALGORITHM23_READY_FOR_CLAIM' if claim_ready else 'DO_NOT_CLAIM_ALGORITHM23_YET', 'num_passed': int(acceptance_df['PASS'].sum()) if len(acceptance_df) else 0, 'num_required': int(len(acceptance_df)), 'PASS': bool(claim_ready)}]))


,gate,required_outcome,observed,PASS
0,CF NLL sanity,"finite, non-constant NLL over q perturbations",ok,True
1,Representation sanity,same q/structure gives stable NLL under expect...,stable_repeat_nll,True
2,Oracle-template q-fit,geometry q-fit reaches good RMSE,best_frac_rmse=0.0056081,True
3,Oracle-template CF-MCMC,CF proposals selected by KLDM survival beat q_...,no_survival_gain,False
4,PyXtal coverage,top-k templates include a good candidate,top3_min_frac_rmse=0.0114551,True
5,Selection comparison,KLDM survival selection beats or matches geome...,no_survival_gain,False
6,Survival score utility,survival score correlates with GT RMSE and top...,survival_correlates_and_selects_well,True
7,Local CPS update,weak late survival-selected CF proposals impro...,no_local_gain,False
8,Full repeated-seed sampling,improves mean or best-of-k over Algorithm22B,proxy_only_not_full_sampler,False


,final_gate,decision,num_passed,num_required,PASS
0,Algorithm23 claim readiness,DO_NOT_CLAIM_ALGORITHM23_YET,5,9,False


In [14]:
focused_keys = [
    'algorithm23_smoke_cf_active',
    'algorithm23_smoke_mcmc_proposal_validity',
    'algorithm23_em_test_cf_sampling_basin',
    'algorithm23_em_test_cf_sampling_basin_epsilon',
    'algorithm23_a1_oracle_qfit_baseline',
    'algorithm23_a2_oracle_template_cf_mcmc_local_improvement',
    'algorithm23_a2_oracle_template_cf_mcmc_summary',
    'algorithm23_a3_oracle_template_cf_mcmc_stress',
    'algorithm23_a3_oracle_template_cf_mcmc_stress_summary',
    'algorithm23_a5_oracle_template_cps_survival',
    'algorithm23_a5_oracle_template_cps_survival_summary',
    'algorithm23_b1_pyxtal_template_enumeration',
    'algorithm23_b2_geometry_qfit_all_templates',
    'algorithm23_b2_geometry_qfit_summary',
    'algorithm23_b3_cf_mcmc_topk_geometry_templates',
    'algorithm23_b3_cf_mcmc_topk_summary',
    'algorithm23_b4_survival_score_vs_rmse',
    'algorithm23_b4_survival_score_vs_rmse_summary',
    'algorithm23_b5_local_cps_update_non_oracle_templates',
    'algorithm23_b5_local_cps_update_summary',
    'algorithm23_c1_repeated_seed_sampler',
    'algorithm23_c2_final_step_only_cf_survival_proxy',
    'algorithm23_c1_pipeline_verdict',
]
summary_rows = []
for key in focused_keys:
    df = result_tables.get(key)
    if df is None:
        summary_rows.append({'table': key, 'rows': 0, 'pass_rate': np.nan})
        continue
    pass_rate = float(df['PASS'].mean()) if 'PASS' in df.columns and len(df) else np.nan
    summary_rows.append({'table': key, 'rows': int(len(df)), 'pass_rate': pass_rate})
display(pd.DataFrame(summary_rows))
verdict_df = result_tables.get('algorithm23_c1_pipeline_verdict', pd.DataFrame())
if verdict_df is not None and len(verdict_df):
    display(verdict_df[['graph','recommendation','algo23_proxy_beats_baseline_frac_rmse_strict','algo23_proxy_beats_algo22_geom_frac_rmse_strict','algo23_proxy_beats_baseline_rmse_strict','algo23_proxy_beats_algo22_geom_rmse_strict']])


,table,rows,pass_rate
0,algorithm23_smoke_cf_active,1,1.0
1,algorithm23_smoke_mcmc_proposal_validity,1,1.0
2,algorithm23_a1_oracle_qfit_baseline,3,1.0
3,algorithm23_a2_oracle_template_cf_mcmc_local_i...,6,1.0
4,algorithm23_a2_oracle_template_cf_mcmc_summary,3,0.0
5,algorithm23_a3_oracle_template_cf_mcmc_stress,4,1.0
6,algorithm23_a3_oracle_template_cf_mcmc_stress_...,1,0.0
7,algorithm23_a5_oracle_template_cps_survival,36,1.0
8,algorithm23_a5_oracle_template_cps_survival_su...,12,0.0
9,algorithm23_b1_pyxtal_template_enumeration,1,1.0


,graph,recommendation,algo23_proxy_beats_baseline_frac_rmse_strict,algo23_proxy_beats_algo22_geom_frac_rmse_strict,algo23_proxy_beats_baseline_rmse_strict,algo23_proxy_beats_algo22_geom_rmse_strict
0,2,promising_local_cf_survival_proxy_only,True,True,True,True
